# 材料別の温度シフト GPR：流動中樹脂温度 Tff を GP の入力に使うかの比較

シフトファクター検討用のノート（同じフォルダの `材料別_温度シフトファクターの検討_ARD対応Matern52＋線形＋ホワイトノイズ.ipynb`）の結果を読み、**流動中樹脂温度 Tff を GP の入力から外したモデル**を学習して、検討のすべてのモデルと比べる（2026-11-06 のユーザーの問い：「Tff はシフトの役目を果たさなくても、入力に使ったほうがよいか」）。学習は 5data、検証は 3data のまま変えない。

**今のモデルの入力**（材料・版によらず同じ。`mat_common.layout`）：

| モデル | GP の入力 | シフトに使う温度 |
|---|---|---|
| B_Tset_* | ln γ̇ ＋ ln a_h(Tset)、ln P3_3、**ln Tff** | Tset（横・縦） |
| A_Tset_* | ln γ̇、ln P3_3、**ln Tff** | Tset（縦だけ） |
| B_Tff_* | ln γ̇ ＋ ln a_h(Tff)、ln P3_3、Tset | Tff（横・縦） |
| A_Tff_* | ln γ̇、ln P3_3、Tset | Tff（縦だけ） |
| C | ln γ̇、ln P3_3、**ln Tff**、Tset | なし |
| **C_noTff**（このノートで学習） | ln γ̇、ln P3_3、Tset | なし |

どのモデルも、Tff を、GP の入力かシフトのどちらかに使っている。**C_noTff** は、C から ln Tff を外したもので、**Tff をまったく使わない**。カーネルは、ほかのモデルと同じ（ARD Matern 5/2 ＋ 線形項 ＋ ノイズ）。C_noTff は、A_Tff・B_Tff の係数を 0 にした点（θ = 0）と同じモデルなので、その対数周辺尤度は、検討のノートの `シフトファクターの推定.csv` の `LML_at_0` にも出ている（3 節で、一致することを確かめる）。ただし、θ = 0 のモデルは保存されていないので、3data の予測と図は、このノートで学習し直して作る。

**比べること**：
- 4 節：**対数周辺尤度（LML）**。C（Tff あり）と C_noTff（Tff なし）の差 = **Tff を入力に足した効果**。検討のほかのモデル（本番の候補を含む）とも並べる。
- 5 節：**3data の誤差**（ln η の RMSE・95% 予測区間に入る割合）。全体・低速側・高速側。
- 6 節：ハイパーパラメータ（長さスケールなど）。7 節：C_noTff の学習結果グラフ。8 節：まとめ（「Tff を入れたほうがよい／どちらでもよい／外したほうがよい」）。

**LML を比べてよい範囲**：同じ材料・同じ版（通常版／温度適切版）・同じ入力の取り方のモデルどうしだけ（目的変数の ln η の行が同じ。シフトは ln η から ln a_v を引く平行移動なので、尤度の大きさは変わらない。中心化は固定の定数）。**材料や版が違うと、LML の大きさはデータの行数とばらつきで決まるので、比べない**（比べるなら、同じ物差しの 3data の誤差か、それぞれの中での差を並べる）。パラメータの数が多いモデルほど LML が大きく出やすいので、差には基準（パラメータ 1 つで 1.92、2 つで 3.0。尤度比検定の 5% の目安。モデルが入れ子でないときは目安）を当てる。

**読むときの注意**：Tff は流動の結果（せん断発熱で射出速度とともに変わる）で、予測のときには事前に分かっている必要がある。このプロジェクトの説明変数（γ34・P3_3・Tff・Tset）はユーザーの指定なので、本番の入力から Tff を外すかは、このノートの結果を見て、ユーザーが決める（外す場合は、学習用のノートに設定を足す別の作業になる）。

## 使い方

**初めての人へ**：1 つ上のフォルダ（`材料別_温度シフトファクター検討/`）の説明書 `説明書_材料別温度シフトGPRのノート.pdf` に、ノートの目的・関係・使えるデータ・使い方をまとめてある。

1. **先にシフトファクター検討用のノート `材料別_温度シフトファクターの検討_ARD対応Matern52＋線形＋ホワイトノイズ.ipynb` を実行しておく**（このノートは、その結果のフォルダ `<学習結果フォルダ>/<版>/全厚み総合/シフトファクターの検討/` の、学習済みの C〔`C_温度シフトなし/学習済みモデル_C.joblib`〕と予測値、表〔`ハイパーパラメータ.csv`・`シフトファクターの推定.csv`・`予測検証_3data_指標.csv`〕を読む）。`INPUT_MODE` は、そのノートと同じにする。
2. メニューの「すべて実行」で、上から順に実行する。重いのは 3 節の学習だけ（材料ごとに GP を 1 回。プロファイル尤度はない）。目安：3 材料で数分〜十数分（要確認）。カーネルは、このプロジェクトの `.venv` を選ぶ。
3. **実行の始めに（「2. 読み込み」のセルで）、材料ごとにダイアログが 1 回開く**：シフトファクター検討用のノートで選んだ**学習結果フォルダ**を選ぶ。学習データ・検証データの場所は、学習済みの C に記録されたものを使う。**通常版と温度適切版は、学習結果フォルダが別なので、版ごとに 1 回ずつ実行する**（2 回目は、温度適切版の学習結果フォルダを選ぶ）。設定 `FILES` に書けば、ダイアログは出ない。ダイアログが見当たらないときは、VS Code の後ろに隠れているので、タスクバーで探す。
4. 各節の最後に、**判定**が出る（`>> 判定：OK` か `>> 判定：★要確認`）。OK は「想定どおり」、★ は「想定と違う、または結果を読むときに注意が要る点」（このノートでは、「Tff を外しても変わらない・良くなる」ことも ★ で知らせる）。★ が出ても、エラーではない。最後の「判定の一覧」に一覧が出る。
5. 図と表は、学習結果フォルダの **`<版>/全厚み総合/Tffの入力の有無の比較/`** に保存される（検討の結果のフォルダ `シフトファクターの検討/` は変えない）。C_noTff の学習済みモデルも、ここに保存する。図は 600 dpi の 1 枚だけ（設定 `FIG_DPI`）。

## 1. 設定

In [ ]:
# セル1：設定エリア  ▼▼▼ 変えてよいのは、ここだけ ▼▼▼
from pathlib import Path      # （下の REPO で、フォルダをたどるのに使う）

SEED = 42                     # 乱数シード（シフトファクター検討用のノートと同じ）

NOISE_LB = 1e-6               # ノイズ分散の下限（シフトファクター検討用のノートと同じにする）

USE_LINEAR = True             # 線形項（DotProduct）を足すか（シフトファクター検討用のノートと同じにする）

INPUT_MODE = "shot"           # 読む結果の入力の取り方（シフトファクター検討用のノートと同じにする。"shot"：ショットごと〔既定。全ショット版/〕、"cond"：条件平均〔条件ごとの平均版/〕）

N_RESTARTS = 20               # C_noTff の学習での、最適化のやり直し回数（シフトファクター検討用のノートの最終モデルと同じ）

USE_EXCLUSIONS = True         # 問題データ一覧（05 フォルダ）の「除外」を除く（シフトファクター検討用のノートと同じ）

REPO = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "04_補助スクリプト").is_dir()))   # リポジトリ直下（このノートの場所から上へたどって、04_補助スクリプト があるフォルダを探す）

MATERIALS = ["PP", "MIX", "GPPS"]    # 調べる材料（PP：バージン PP、MIX：粉砕材混合 PP、GPPS）。材料ごとに、実行の始めにダイアログで、学習結果フォルダ（シフトファクター検討用のノートで選んだもの）を選ぶ

MAT_FOLDERS = {"PP": "01_バージンPP再々実験(データと結果)", "MIX": "02_粉砕材混合PP実験(データと結果)", "GPPS": "03_GPPS実験(データと結果)"}   # 05 フォルダの中の材料フォルダ名（ダイアログの最初のフォルダ）

FILES = {}                    # ダイアログを使わずに決めるとき：{"PP": {"結果": r"…"}}（相対パスはリポジトリ直下から。"学習"・"検証" も書くと、学習済みの C の記録より、そちらを使う）
SAVE_ROOT = None              # 結果の置き場所：None＝学習結果フォルダの <版>/全厚み総合/Tffの入力の有無の比較/（既定）／フォルダを書く＝その下の <材料>/<版>/全厚み総合/Tffの入力の有無の比較/（確かめの実行用）
N_JOBS = 8                    # 材料ごとの学習を並列に計算する数（CPU の数に合わせる）
FIG_DPI = 600                 # 保存する図の解像度（すべての図を、この解像度の 1 枚だけで保存する）
SHOW_FIGS = True              # True：図をノートにも表示する（False：保存だけ）
# ---- ここから下は、このノートだけの設定 ----
LML_CLEAR = 1.92              # 4・8 節：パラメータ 1 つの差の基準（尤度比検定の 5% の目安。C と C_noTff の差は、長さスケール 1 つ分）。2 つ以上の差は、カイ二乗分布の 95% 点 ÷ 2 を使う
RMSE_RATIO = 1.1              # 5・8 節：3data の RMSE の比が、この倍率の内側（1/1.1〜1.1 倍）なら「同等」（シフトファクター検討用のノートの「C の 1.1 倍以内」と同じ考え方）
LML_CHECK = 1.0               # 3 節：C_noTff の LML が、検討のノートの θ = 0 の値（A_Tff・B_Tff の LML_at_0）より、これ以上低ければ ★（最適化の局所解の目安）
# ▲▲▲ 設定エリア終了 ▲▲▲

In [ ]:
# セル2：ライブラリ、フォント
%matplotlib inline
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.tri as mtri
from IPython.display import display

plt.rcParams["figure.dpi"] = 80
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
try:
    fm.findfont("IPAexGothic", fallback_to_default=False)
except ValueError:
    raise RuntimeError("日本語フォント IPAexGothic が見つかりません。インストールしてからカーネルを再起動してください。")
plt.rcParams["font.family"] = ["IPAexGothic", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("Python:", sys.executable, "/", sys.version.split()[0])
print("材料:", MATERIALS, "／入力の取り方:", INPUT_MODE)

### 共通の部品

シフトファクター検討用のノートと同じ部品（`04_補助スクリプト/mat_common.py`）。

In [ ]:
# セル：共通の部品（読み込み・シフトファクター・モデル・評価・図）。04_補助スクリプト/mat_common.py の中身。直すときは、そのファイルを直して、ノートを作り直す
import glob
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, DotProduct, Matern, Sum, WhiteKernel
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------- 設定（ノートの設定エリアの値があれば、それを使う）
SEED = globals().get("SEED", 42)
NOISE_LB = globals().get("NOISE_LB", 1e-6)                  # ノイズ分散の下限（6月のノートは 1e-3）
N_RESTARTS = globals().get("N_RESTARTS", 20)                # 最終モデルの学習での、最適化のやり直し回数（ハイパーパラメータの局所解が多いので、6月の 5 回より多くする）
PROFILE_RESTARTS = globals().get("PROFILE_RESTARTS", 1)     # プロファイル尤度の格子の各点での、やり直し回数（前の点の解も初期値に使う）
USE_LINEAR = globals().get("USE_LINEAR", True)              # True：6月のカーネルに線形項（DotProduct）を足す
USE_EXCLUSIONS = globals().get("USE_EXCLUSIONS", True)      # 問題データ一覧の「除外」を使うか
REPO = globals().get("REPO", os.environ.get("NB_REPO", ".."))
N_JOBS = globals().get("N_JOBS", 8)
FIG_DPI = globals().get("FIG_DPI", 600)                     # 保存する図の解像度。すべての図を、この解像度の 1 枚だけで保存する（報告書・発表資料用。2026-10-27 から）
SHOW_FIGS = globals().get("SHOW_FIGS", True)                # 図をノートに表示するか（False なら保存だけ）
INPUT_MODE = globals().get("INPUT_MODE", "shot")            # "shot"：γ̇・P3_3・Tff をショットごとの値で入れる（6月のノートと同じ。既定）／"cond"：条件（設定温度・厚み・射出速度）ごとの平均で入れる（参考）
MAX_REFINE = globals().get("MAX_REFINE", 4)                 # プロファイル尤度で、最良の点が細かい格子の端に当たったとき、その点を中心に細かい格子を取り直す回数の上限
MAX_EXTEND = globals().get("MAX_EXTEND", 20)                # プロファイル尤度で、95% 区間の端が細かい格子の中で決まるまで、区間の外側に細かい格子の点を 1 列ずつ足す回数の上限（2026-11-03）
FINE_B = globals().get("FINE_B", (4.0, 1.0))               # 構造 B の細かい格子：最良の点の ±4 を 1 刻み
FINE_A = globals().get("FINE_A", (4.5, 0.5))                # 構造 A の細かい格子：最良の点の ±4.5 を 0.5 刻み（粗い格子の刻み 5 の、隣の点の手前まで。2026-11-02）
OUTLIER_Z = globals().get("OUTLIER_Z", 3.5)                 # 規則 f：|z| がこれを超え、かつ |残差|（ln η）が次の値を超える条件
OUTLIER_RESID = globals().get("OUTLIER_RESID", 0.1)
P33_DEV = globals().get("P33_DEV", 0.2)                     # 規則 g：P3_3 が条件の中央値から、この割合以上離れたショットを候補にする
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # 下限・上限への張り付きは、ハイパーパラメータの表に記録する

# ---------------------------------------------------------------- データの場所と列
DATA_ROOT = os.path.join(REPO, "05_材料ごとの機械学習データと予測検証データ")
PROBLEM_CSV = os.path.join(DATA_ROOT, "問題データ一覧.csv")
MAT_FOLDERS = globals().get("MAT_FOLDERS", {"PP": "01_バージンPP再々実験", "MIX": "02_粉砕材混合PP実験", "GPPS": "03_GPPS実験"})
MAT_NAMES = {"PP": "PP", "MIX": "粉砕材混合PP", "GPPS": "GPPS"}     # 問題データ一覧の「材料」列の名前
DATA_SUB = "*_学習・検証データ"                                     # 材料フォルダの中の、学習データと検証データのフォルダをまとめたフォルダ（2026-11-03 のユーザーのフォルダの整理〔a4ad4b8〕で、01_学習・検証データ/ などができた）
GROUP_DIRS = {"5data": "学習データ_*", "3data": "検証データ_*"}      # DATA_SUB の中の、学習データと検証データのフォルダ（CSV が 1 つずつ）
COLS = {"元ファイル": "file", "キャビ厚": "H", "射出速度[mm/s]": "v", "樹脂粘度η34(Pa・s)": "eta", "せん断速度γ34(1/s)": "gd",
        "流動中樹脂圧力P3_1(MPa)": "P31", "流動中樹脂圧力P3_2(MPa)": "P32", "流動中樹脂圧力P3_3(MPa)": "P33",
        "加熱シリンダ温度[℃]": "Tset", "加熱シリンダ温度(℃)": "Tset", "流動中樹脂温度(℃)": "Tff"}
REQUIRED = ["file", "H", "v", "eta", "gd", "P33", "Tff"]          # 加熱シリンダ温度の列がなければ、元ファイル名から取る
RE_SRC = r"^(\d{3})_(\d+)℃_(\d{3})_([\d.]+)mm[A-Za-z]*\.csv$"     # ショット_設定温度_射出速度_厚み（GPPS は末尾に NR が付く）
R_GAS = 8.314                                                      # 気体定数 [J/mol/K]
D = {}             # (材料, 組) → DataFrame。組は "5data"（学習）と "3data"（検証）
FILE_INFO = {}     # (材料, 組) → 読んだファイルの情報
REMOVED = {}       # (材料, 組) → 一覧の「除外」で除いた行（元ファイルと ID）
INVALID = {}       # (材料, 組) → 0 以下の値（η・γ̇・P3_3）で対数がとれず、一覧に登録がないまま除いた行
JUDGES = []        # 判定の記録（まとめの表に使う）
SOURCES = {}       # 材料 → {"5data": 学習データの CSV, "3data": 検証データの CSV, "結果": 学習結果フォルダ}（choose_inputs で決める）

# ---------------------------------------------------------------- 結果の置き場所（2026-10-27 からの構成）
VERSION_DIRS = {"cond": "条件ごとの平均版", "shot": "全ショット版"}   # 学習結果フォルダの直下の分け方（入力の取り方 INPUT_MODE ごと）。図のタイトルと、表の「入力」の列にも使う
THICK_TOTAL, THICK_EACH = "全厚み総合", "厚みごと"   # その下の分け方：全厚みの 5data で 1 つのモデルを学習した結果（学習用・シフトファクター検討用のノート）／厚みごとの比較のノートの結果
STUDY_DIR = "シフトファクターの検討"                  # 全厚み総合/ の下の、シフトファクター検討用ノートの結果のフォルダ
MODE_JA = {"shot": "ショットごとの入力", "cond": "条件平均の入力"}   # 10-26 までの結果のフォルダの名前（<材料フォルダ>/<番号>_学習結果/<材料フォルダ>_学習結果(5all学習データ)/ の下。その結果は、この構成のまま残す。発表資料・ポスターのスクリプトが読む）
DATA_KINDS = {"5data": ("学習", "学習データ（5data）の CSV"), "3data": ("検証", "検証データ（3data）の CSV")}   # 組 → (設定 FILES での名前, ダイアログの説明)
PATH_MAX = 260     # Windows のパスの長さの上限（文字）
DEEPEST = 105      # 学習結果フォルダの下の、最も長いファイルの場所の長さ（文字）の目安（せん断速度_粘度_拡張版/ の英語の凡例の図など。2026-11-02 にモデルのフォルダの名前に型が入って長くなった）


def show_path(p):
    """表や記録に書く場所：リポジトリの中なら、リポジトリ直下からの相対パス。外なら、そのまま（絶対パス）"""
    a, r = os.path.abspath(p), os.path.abspath(REPO)
    try:
        inside = os.path.normcase(os.path.commonpath([a, r])) == os.path.normcase(r)
    except ValueError:                                       # ドライブが違う
        inside = False
    return os.path.relpath(a, r) if inside else a


def full_path(p):
    """設定や記録に書いた場所 → 絶対パス（相対パスは、リポジトリ直下から）"""
    return os.path.abspath(p if os.path.isabs(p) else os.path.join(REPO, p))


def ask_path(title, kind="file", initialdir=None):
    """ダイアログで、CSV ファイル（kind = "file"）かフォルダ（kind = "dir"）を選ぶ（tkinter）。選んだ場所を返す。
    キャンセルしたら止める（例外）。ダイアログは、ほかのウィンドウの手前に出す（VS Code の後ろに隠れたら、タスクバーで探す）"""
    import tkinter as tk
    from tkinter import filedialog
    print(f"ダイアログを開きます：{title}", flush=True)
    root = tk.Tk()
    root.withdraw()                                          # 空の親ウィンドウは見せない
    root.attributes("-topmost", True)                       # ダイアログを、ほかのウィンドウの手前に出す
    root.update()
    start = initialdir if initialdir and os.path.isdir(initialdir) else None
    try:
        if kind == "file":
            p = filedialog.askopenfilename(parent=root, title=title, initialdir=start, filetypes=[("CSV ファイル", "*.csv"), ("すべてのファイル", "*.*")])
        else:
            p = filedialog.askdirectory(parent=root, title=title, initialdir=start, mustexist=False)
    finally:
        root.destroy()
    if not p:
        raise RuntimeError(f"選ばれなかったので止めました（{title}）。もう一度、このセルから実行してください")
    return os.path.normpath(p)


def choose_inputs(mats, files=None, need_data=True):
    """材料ごとに、学習データ（5data）・検証データ（3data）の CSV と、学習結果フォルダを決めて、SOURCES に記録する（実行の始めに、まとめて聞く）。
    files（ノートの設定 FILES）：{材料: {"学習": CSV, "検証": CSV, "結果": フォルダ}}。書いてあるものはそれを使い、ないものはダイアログで選ぶ（相対パスは、リポジトリ直下から）。
    need_data = False なら、学習結果フォルダだけを決める（データの場所は、学習済みモデルの記録から取る。data_from_model）。
    同じ学習結果フォルダを 2 つの材料で選ぶと、結果が上書きされるので止める。決めたものの表を返す"""
    files = files or {}
    rows = []
    for mat in mats:
        f, src = files.get(mat, {}), SOURCES.setdefault(mat, {})
        start = os.path.join(DATA_ROOT, MAT_FOLDERS.get(mat, ""))   # ダイアログの最初のフォルダ（05 フォルダの材料フォルダ）
        for grp, (name, what) in DATA_KINDS.items():
            if not (f.get(name) or need_data):
                continue
            p = full_path(f[name]) if f.get(name) else ask_path(f"【{MAT_NAMES[mat]}】{what} を選んでください", "file", start)
            if not os.path.isfile(p):
                raise FileNotFoundError(f"{MAT_NAMES[mat]}：{what} が見つかりません（{p}）")
            src[grp] = p
        d = full_path(f["結果"]) if f.get("結果") else ask_path(
            f"【{MAT_NAMES[mat]}】学習結果フォルダを選んでください（この下に「{VERSION_DIRS['shot']}」「{VERSION_DIRS['cond']}」などを作ります）", "dir", start)
        os.makedirs(d, exist_ok=True)
        src["結果"] = d
        if len(os.path.abspath(d)) + DEEPEST > PATH_MAX:
            print(f"★ 注意：{MAT_NAMES[mat]} の学習結果フォルダの場所が長い（{len(os.path.abspath(d))} 文字）ので、中に作るファイルの場所が、"
                  f"Windows の上限 {PATH_MAX} 文字を超えるおそれがあります。もっと浅いフォルダを選ぶと安全です")
        rows.append({"材料": MAT_NAMES[mat], **{f"{DATA_KINDS[g][0]}データ": show_path(src[g]) for g in DATA_KINDS if src.get(g)}, "学習結果フォルダ": show_path(d)})
    roots = [os.path.normcase(os.path.abspath(SOURCES[m]["結果"])) for m in mats]
    dup = sorted({r for r in roots if roots.count(r) > 1})
    if dup:
        raise ValueError("2 つ以上の材料で、同じ学習結果フォルダが選ばれました（結果が上書きされます）。材料ごとに別のフォルダを選んでください：" + "、".join(dup))
    return pd.DataFrame(rows)


def data_from_model(mat, obj):
    """学習済みモデル（joblib の中身 obj）に記録した、学習データ・検証データの場所を SOURCES に入れる（設定 FILES で決めたものは、そのまま使う）。
    記録の場所にファイルがなければ、ダイアログで選ぶ"""
    src = SOURCES.setdefault(mat, {})
    for grp, (name, what) in DATA_KINDS.items():
        if src.get(grp):
            continue
        rec = obj.get(f"{name}データのファイル")
        p = full_path(rec) if rec else None
        if not (p and os.path.isfile(p)):
            print(f"【{MAT_NAMES[mat]}】学習済みモデルに記録された{what} が見つかりません（{rec}）")
            p = ask_path(f"【{MAT_NAMES[mat]}】{what} を選んでください", "file", os.path.join(DATA_ROOT, MAT_FOLDERS.get(mat, "")))
        src[grp] = p


def result_root(mat):
    """学習結果フォルダ（choose_inputs で決めたもの）"""
    d = SOURCES.get(mat, {}).get("結果")
    if not d:
        raise RuntimeError(f"{MAT_NAMES[mat]}：学習結果フォルダが決まっていません（先に choose_inputs で決める）")
    return d


def result_dir(mat, *parts, make=True):
    """学習結果フォルダの下の場所。parts の中のモデルの記号（B_Tset など）は、フォルダの名前（MODEL_DIRS）に直す。
    make が True なら、フォルダがなければ作る（読むだけのときは False）。
    例：result_dir(mat, "データ確認")、result_dir(mat, VERSION_DIRS[INPUT_MODE], THICK_TOTAL, "B_Tset")"""
    d = os.path.join(result_root(mat), *[MODEL_DIRS.get(p, p) for p in parts])
    if make:
        os.makedirs(d, exist_ok=True)
    return d


def path_of(mat, grp):
    """読む CSV：choose_inputs で決めたもの。決めていなければ（ノートの外のスクリプト）、05 フォルダの材料フォルダの DATA_SUB の中から探す（2026-11-03 から。
    それまでは材料フォルダの直下）。そのときは、名前に「温度適切」を含むフォルダ（2026-11-02 に増えた温度適切版）を除き、通常版（10-26 までの結果に使ったデータ）を使う"""
    if SOURCES.get(mat, {}).get(grp):
        return SOURCES[mat][grp]
    fs = [f for f in sorted(glob.glob(os.path.join(DATA_ROOT, MAT_FOLDERS[mat], DATA_SUB, GROUP_DIRS[grp], "*.csv"))) if "温度適切" not in os.path.basename(os.path.dirname(f))]
    assert len(fs) == 1, f"{MAT_FOLDERS[mat]}/{DATA_SUB}/{GROUP_DIRS[grp]} に CSV が 1 つだけあるはず（見つかったもの: {fs}）"
    return fs[0]


# ---------------------------------------------------------------- 問題データ一覧
def read_problem_list(path=None):
    """問題データ一覧.csv（05 フォルダ）を読む（空欄は空文字）"""
    return pd.read_csv(path or PROBLEM_CSV, encoding="utf-8-sig", dtype=str).fillna("")


def problem_mask(df, mat, grp, plist, kinds):
    """区分が kinds のどれかで、再測定が「済み」でない一覧の行のうち、この材料・組・条件に当てはまる行の真偽値と、その ID を返す。
    「ショット」列は、空欄なら条件全体、数字ならそのショット（元ファイル名の先頭 3 桁）。"2;3" のように複数書ける"""
    mask = np.zeros(len(df), bool)
    ids = [[] for _ in range(len(df))]
    for _, r in plist.iterrows():
        if r["区分"] not in kinds or r["再測定"] == "済み" or r["データ"] not in ("新", ""):
            continue
        if r["材料"] not in ("すべて", "", MAT_NAMES[mat]) or r["組"] not in ("", grp):
            continue
        m = np.ones(len(df), bool)
        if r["設定温度"] != "":
            m &= df["Tset"].to_numpy() == float(r["設定温度"])
        if r["厚み"] != "":
            m &= np.isclose(df["H"].to_numpy(), float(r["厚み"]))
        if r["射出速度"] != "":
            m &= df["v"].to_numpy() == float(r["射出速度"])
        if r.get("ショット", "") != "":
            shots = [int(s) for s in re.split(r"[;；,、]", r["ショット"]) if s.strip()]
            m &= np.isin(df["shot"].to_numpy(), shots) if "shot" in df.columns else np.zeros(len(df), bool)
        mask |= m
        for i in np.flatnonzero(m):
            ids[i].append(r["ID"])
    return mask, [";".join(x) for x in ids]


# ---------------------------------------------------------------- 読み込み
def load(mat, grp, plist=None, exclude=None):
    """結合ファイル（path_of：choose_inputs で選んだ CSV）を読み、形式を確かめ、問題データ一覧の「除外」を除く。
    加熱シリンダ温度の列がないファイル（PP の 3data）は、元ファイル名から設定温度を取る。列があれば、ファイル名と一致するかを確かめる。
    η・γ̇・P3_3 が 0 以下の行は対数がとれないので、一覧に登録がなくても除き、INVALID に記録する（判定で ★）。
    列 sens には「感度確認」の ID が入る（なければ空文字）。圧力は Pa に直してから ln をとる。Tff の ln は K で"""
    exclude = USE_EXCLUSIONS if exclude is None else exclude
    path = path_of(mat, grp)
    raw = pd.read_csv(path, encoding="utf-8-sig")
    unknown = [c for c in raw.columns if c not in COLS]
    assert not unknown, f"想定にない列がある: {unknown}（mat_common.py の COLS に足す）"
    df = raw.rename(columns=COLS)
    missing = [c for c in REQUIRED if c not in df.columns]
    assert not missing, f"必要な列がない: {missing}（{path}）"
    assert df[REQUIRED].isna().sum().sum() == 0, f"欠損がある（{path}）"
    p = df["file"].str.extract(RE_SRC)
    assert p.notna().all().all(), f"元ファイル名の形式に合わない行がある（{path}）"
    assert np.allclose(p[2].astype(float), df["v"]) and np.allclose(p[3].astype(float), df["H"]), "元ファイル名と、厚み・射出速度の列が一致しない"
    has_tset = "Tset" in df.columns
    if has_tset:
        assert np.allclose(p[1].astype(float), df["Tset"].astype(float)), "元ファイル名と、加熱シリンダ温度の列が一致しない"
    else:
        df["Tset"] = p[1].astype(float)
    df["Tset"] = df["Tset"].astype(float)
    df["shot"] = p[0].astype(int)
    plist = read_problem_list() if plist is None else plist
    n_raw = len(df)
    if exclude:
        drop, ids = problem_mask(df, mat, grp, plist, ("除外",))
        REMOVED[(mat, grp)] = pd.DataFrame({"元ファイル": df.loc[drop, "file"].to_numpy(), "ID": np.array(ids, dtype=object)[drop]})
        df = df[~drop].reset_index(drop=True)
    else:
        REMOVED[(mat, grp)] = pd.DataFrame(columns=["元ファイル", "ID"])
    bad = (df[["eta", "gd", "P33"]] <= 0).any(axis=1).to_numpy()
    INVALID[(mat, grp)] = df.loc[bad, ["file", "Tset", "H", "v", "shot", "eta", "gd", "P33"]].reset_index(drop=True)
    df = df[~bad].reset_index(drop=True)
    _, df["sens"] = problem_mask(df, mat, grp, plist, ("感度確認",))
    df["is_sens"] = df["sens"] != ""
    df["cond"] = df["Tset"].astype(int).astype(str) + "_" + df["H"].astype(str) + "_" + df["v"].astype(int).astype(str)
    df["ln_eta"], df["ln_gd"], df["ln_H"], df["ln_v"] = np.log(df["eta"]), np.log(df["gd"]), np.log(df["H"]), np.log(df["v"])
    df["ln_P33"] = np.log(df["P33"] * 1e6)                 # CSV の圧力は MPa。Pa に直してから ln をとる
    df["ln_Tff"] = np.log(df["Tff"] + 273.15)               # 流動中樹脂温度 [K] の ln（6月のノートと同じ）
    for c in ["gd", "P33", "Tff"]:                          # 条件ごとの平均（INPUT_MODE = "cond" の入力、図の予測線、外れ値の検知に使う）
        df[f"ln_{c}_cm"] = df.groupby("cond")[f"ln_{c}"].transform("mean")
    df["Tff_cm"] = np.exp(df["ln_Tff_cm"]) - 273.15
    df["grp"] = grp
    FILE_INFO[(mat, grp)] = {"材料": MAT_NAMES[mat], "組": grp, "ファイル": show_path(path), "読んだ行数": n_raw,
                             "一覧で除外": n_raw - len(df) - int(bad.sum()), "0以下で除外（未登録）": int(bad.sum()), "使う行数": len(df),
                             "条件数": df["cond"].nunique(), "ショット/条件": f"{df.groupby('cond').size().min()}〜{df.groupby('cond').size().max()}",
                             "設定温度": "・".join(str(int(t)) for t in sorted(df["Tset"].unique())), "設定温度の取り方": "列" if has_tset else "元ファイル名",
                             "厚み": "・".join(f"{h:g}" for h in sorted(df["H"].unique())), "射出速度": "・".join(str(int(v)) for v in sorted(df["v"].unique())),
                             "感度確認の行": int(df["is_sens"].sum())}
    return df


def load_all(mats, exclude=None):
    """材料ごとに 5data（学習）と 3data（検証）を読む。学習と検証で条件が重ならないことも確かめる"""
    plist = read_problem_list()
    out = {(mat, grp): load(mat, grp, plist, exclude) for mat in mats for grp in ["5data", "3data"]}
    for mat in mats:
        both = set(out[(mat, "5data")]["cond"]) & set(out[(mat, "3data")]["cond"])
        assert not both, f"{MAT_NAMES[mat]}：学習と検証で条件が重なっている: {sorted(both)}"
    return out


def add_groups(tr, te):
    """検証の行を、学習データの射出速度の範囲より遅い側（低速側）・速い側（高速側）・範囲内に分ける（列 group）"""
    te = te.copy()
    vmin, vmax = tr["v"].min(), tr["v"].max()
    te["group"] = np.where(te["v"] < vmin, "低速側", np.where(te["v"] > vmax, "高速側", "範囲内"))
    return te


# ---------------------------------------------------------------- 温度のシフトファクター
# 2026-11-02（ユーザー決定）：型を材料ごとに分けず、どの材料にも両方の型を当てはめられるようにした。材料ごとの設定 SHIFT_CFG は基準温度と WLF 型の Tg、
# 型ごとの設定 FORM_CFG は係数を調べる格子。モデルに使う設定（cfg）は、2 つを合わせたもの（shift_cfg）
FORMS = {"arrhenius": "Arr", "wlf": "WLF"}          # シフトファクターの型 → モデルの記号の末尾（例：B_Tset_Arr）
FORM_JA = {"arrhenius": "アレニウス型", "wlf": "WLF型"}
DEFAULT_SHIFT_CFG = {     # 材料ごとの基準温度 Tref [℃]（None なら中央の設定温度）と、WLF 型の D2 = Tg [℃]（PP・粉砕材混合 PP の −10 ℃ は、Moldflow の PP でよく使われる D2 = 263.15 K。GPPS の 100 ℃ とも仮置き）
    "PP": {"Tref": 210.0, "Tg": -10.0},
    "MIX": {"Tref": 210.0, "Tg": -10.0},
    "GPPS": {"Tref": 220.0, "Tg": 100.0},
}
DEFAULT_FORM_CFG = {      # 型ごとの、尤度を調べる粗い格子（下限, 上限, 刻み。h が横、v が縦のシフトの係数）と、細かい格子 fine（±幅, 刻み。書いた型だけ、FINE_A・FINE_B の代わりに使う）
    "arrhenius": {"h_grid": (-20.0, 40.0, 5.0), "v_grid": (-20.0, 60.0, 5.0)},                            # θ = E [kJ/mol]。細かい格子は FINE_B・FINE_A
    "wlf": {"A2": 51.6, "h_grid": (-20.0, 40.0, 5.0), "v_grid": (-20.0, 40.0, 5.0), "fine": (2.0, 0.25)},   # θ = A1 [−]、A2 [K] は固定。A1 の 1 は、粘度の比にして E の 1 kJ/mol より大きい（GPPS で約 3.5 倍）ので、0.25 刻み
}
SHIFT_CFG = globals().get("SHIFT_CFG", DEFAULT_SHIFT_CFG)
FORM_CFG = globals().get("FORM_CFG", DEFAULT_FORM_CFG)


def shift_cfg(mat, form, tr=None):
    """材料 mat に型 form（"arrhenius"・"wlf"）を当てはめるときの設定（shift_ln・run_profile などに渡す cfg）。Tref が None なら、学習データ tr の中央の設定温度"""
    if form not in FORMS:
        raise ValueError(f"シフトファクターの型は {list(FORMS)} のどれかです（指定：{form}）")
    cfg = {**FORM_CFG[form], **SHIFT_CFG[mat], "type": form}
    if cfg.get("Tref") is None:
        lv = sorted(tr["Tset"].unique())
        cfg["Tref"] = float(lv[len(lv) // 2])
    return cfg


def key_form(key):
    """モデルの記号 → シフトファクターの型（末尾が Arr・WLF の記号だけ。C と、型の付かない 10-26 までの記号〔B_Tset など〕は None）"""
    p = str(key).split("_")
    return next((f for f, s in FORMS.items() if len(p) == 3 and p[2] == s), None)


def shift_ln(T_c, theta, cfg):
    """ln a(T)（T は ℃。Tref で 0）。
    アレニウス型：ln a = (E/R)(1/T − 1/Tref)、theta = E [kJ/mol]。
    WLF 型：ln a = −A1·[(T − D2)/(A2 + T − D2) − (Tref − D2)/(A2 + Tref − D2)]、theta = A1 [−]、D2 = Tg、A2 は固定"""
    T = np.asarray(T_c, float) + 273.15
    Tr = float(cfg["Tref"]) + 273.15
    if cfg["type"] == "arrhenius":
        return float(theta) * 1000.0 / R_GAS * (1.0 / T - 1.0 / Tr)
    D2, A2 = float(cfg["Tg"]) + 273.15, float(cfg["A2"])
    return -float(theta) * ((T - D2) / (A2 + T - D2) - (Tr - D2) / (A2 + Tr - D2))


def theta_names(cfg):
    """(横の名前, 縦の名前, 単位)"""
    return ("E_h", "E_v", "kJ/mol") if cfg["type"] == "arrhenius" else ("A1_h", "A1_v", "−")


# ---------------------------------------------------------------- モデル
MODEL_JA = {"B_Tset": "B_Tset（設定温度 Tset で横・縦にシフト）", "B_Tff": "B_Tff（流動中樹脂温度 Tff で横・縦にシフト）",
            "A_Tset": "A_Tset（Tset で縦だけシフト）", "A_Tff": "A_Tff（Tff で縦だけシフト）",
            "C": "C（シフト構造なし。4 変数を並べた通常の GPR）"}
MODEL_DIRS = {"B_Tset": "B_Tset_温度シフトに加熱シリンダ温度採用", "B_Tff": "B_Tff_温度シフトに型内樹脂温度採用",
              "A_Tset": "A_Tset_縦だけの温度シフトに加熱シリンダ温度採用", "A_Tff": "A_Tff_縦だけの温度シフトに型内樹脂温度採用",
              "C": "C_温度シフトなし"}     # モデルの結果のフォルダの名前（記号_学習方法の短い説明。型内樹脂温度＝流動中樹脂温度 Tff。2026-10-27 のユーザー依頼）
for _k in ["B_Tset", "B_Tff", "A_Tset", "A_Tff"]:      # 型の付いた記号（2026-11-02。例：B_Tset_Arr）。型の付かない記号は、10-26 までの結果を読むスクリプトのために残す
    for _f, _s in FORMS.items():
        MODEL_JA[f"{_k}_{_s}"] = f"{_k}_{_s}" + MODEL_JA[_k][len(_k):-1] + f"。{FORM_JA[_f]}）"
        MODEL_DIRS[f"{_k}_{_s}"] = f"{_k}_{_s}_{FORM_JA[_f]}の" + MODEL_DIRS[_k].split("_", 2)[2]
COL_JA = {"ln_gd": "ln γ̇", "ln_P33": "ln P3_3", "ln_Tff": "ln Tff", "Tset": "Tset"}


def shift_key(col, struct="B", form=None):
    """シフトファクターに使うデータ列 → モデルの記号。col：学習ファイルの列名（"加熱シリンダ温度[℃]"・"加熱シリンダ温度(℃)"・"流動中樹脂温度(℃)"）か、
    短い名前（"Tset"・"Tff"）。None（または空の文字）なら、シフトなしの "C"。struct："B"（横・縦を別々にずらす）か "A"（縦だけ）。温度でない列はエラー。
    form：シフトファクターの型（"arrhenius"・"wlf"）。書くと、記号の末尾に型が付く（B_Tset_Arr など。2026-11-02）"""
    if col is None or str(col).strip() == "":
        return "C"
    t = COLS.get(str(col).strip(), str(col).strip())
    if t not in ("Tset", "Tff"):
        raise ValueError(f"シフトファクターに使える列は、温度の列（加熱シリンダ温度・流動中樹脂温度）だけです（指定された列：{col}）")
    if struct not in ("A", "B"):
        raise ValueError(f"シフトの形は \"B\"（横・縦を別々）か \"A\"（縦だけ）です（指定：{struct}）")
    if form is not None and form not in FORMS:
        raise ValueError(f"シフトファクターの型は {list(FORMS)} のどれかです（指定：{form}）")
    return f"{struct}_{t}" + (f"_{FORMS[form]}" if form else "")


def model_keys(cols, structs=("B",), forms=(None,)):
    """比べるモデルの記号の一覧（シフトの形 structs、列 cols、型 forms の組み合わせ。同じものは 1 つにする）"""
    out = []
    for s in structs:
        for c in cols:
            for f in forms:
                k = shift_key(c, s, f)
                if k not in out:
                    out.append(k)
    return out


def layout(key):
    """GP の入力の列（シフト前の名前）と、シフトに使う温度の列（C は None）。
    B_Tset・A_Tset：(ln γ̇ + ln a_h(Tset), ln P3_3, ln Tff)、B_Tff・A_Tff：(ln γ̇ + ln a_h(Tff), ln P3_3, Tset)、C：(ln γ̇, ln P3_3, ln Tff, Tset)"""
    if key == "C":
        return ["ln_gd", "ln_P33", "ln_Tff", "Tset"], None
    temp = key.split("_")[1]
    return (["ln_gd", "ln_P33", "ln_Tff"], "Tset") if temp == "Tset" else (["ln_gd", "ln_P33", "Tset"], "Tff")


def model_title(key, mode=None):
    """図のタイトルに使うモデルの名前（版〔入力の取り方〕つき）"""
    return f"{MODEL_JA[key]}・{VERSION_DIRS[INPUT_MODE if mode is None else mode]}"


def input_labels(key):
    cols, tcol = layout(key)
    lab = [COL_JA[c] for c in cols]
    if tcol is not None and key.startswith("B"):
        lab[0] = f"ln γ̇ + ln a_h({tcol})"
    return lab


def kernel_for(d, linear=None):
    """6月のノートのカーネル（ARD Matern 5/2 ＋ ホワイトノイズ）。linear=True なら線形項（DotProduct）を足す（射出速度の両端の外挿のため）"""
    linear = USE_LINEAR if linear is None else linear
    k = C(1.0, (1e-3, 1e3)) * Matern(length_scale=[1.0] * d, length_scale_bounds=(1e-2, 1e2), nu=2.5)
    if linear:
        k = k + C(1.0, (1e-3, 1e3)) * DotProduct(sigma_0=1.0, sigma_0_bounds=(1e-5, 1e3))
    return k + WhiteKernel(noise_level=0.1, noise_level_bounds=(NOISE_LB, 1e1))


def mode_col(df, c, mode):
    """入力の列名。mode = "cond" で、条件平均の列（<名前>_cm）があればそれを使う（予測用に組み立てた表には _cm がないので、そのまま）"""
    return f"{c}_cm" if mode == "cond" and f"{c}_cm" in df.columns else c


def design(df, key, cfg, th=0.0, tv=0.0, mode=None):
    """データ → (標準化の定数を決めるための入力 X0〔シフト前〕、実際の入力 Xs〔シフト後〕、出力のシフト量 ysh = ln a_v(T))。
    mode（既定 INPUT_MODE）が "cond" なら、γ̇・P3_3・Tff は条件ごとの平均を使う"""
    mode = INPUT_MODE if mode is None else mode
    cols, tcol = layout(key)
    X0 = np.column_stack([df[mode_col(df, c, mode)].to_numpy(float) for c in cols])
    Xs, ysh = X0.copy(), np.zeros(len(df))
    if tcol is not None:
        T = df[mode_col(df, tcol, mode)].to_numpy(float)
        Xs[:, 0] = X0[:, 0] + shift_ln(T, th, cfg)
        ysh = shift_ln(T, tv, cfg)
    return X0, Xs, ysh


class LossRecorder:
    """6月のノートのカスタムオプティマイザと同じ（L-BFGS-B）。各ステップの損失（負の対数周辺尤度）を、最適化 1 回ごとに runs に記録する"""

    def __init__(self):
        self.runs = []

    def __call__(self, obj_func, initial_theta, bounds):
        from scipy.optimize import minimize
        hist = []
        res = minimize(obj_func, initial_theta, bounds=bounds, method="L-BFGS-B", jac=True,
                       callback=lambda xk: hist.append(float(obj_func(xk, eval_gradient=False))))
        self.runs.append(hist)
        return res.x, res.fun


def fit_gp(key, tr, cfg, th=0.0, tv=0.0, restarts=None, theta0=None, record=False, mode=None):
    """学習。標準化・中心化は、シフトの量によらない定数（シフト前の入力の平均・標準偏差、ln η の平均。学習データだけで計算）。
    出力の自動標準化（normalize_y）は使わない（シフトの量ごとに尤度が比べられなくなるため）。theta0：カーネルの初期値（最初の最適化はここから、
    残りの restarts 回は、探索範囲の中の乱数の初期値から。対数周辺尤度が最大の解を採る）。record=True なら、損失の推移を戻り値の loss_runs に入れる。
    mode：入力の取り方（既定 INPUT_MODE。"shot" か "cond"）。予測にも同じ取り方を使う"""
    mode = INPUT_MODE if mode is None else mode
    X0, Xs, ysh = design(tr, key, cfg, th, tv, mode)
    mu_x, sd_x = X0.mean(axis=0), X0.std(axis=0)
    y = tr["ln_eta"].to_numpy()
    y0 = float(y.mean())
    k = kernel_for(X0.shape[1])
    if theta0 is not None:
        k = k.clone_with_theta(theta0)
    rec = LossRecorder() if record else None
    gp = GaussianProcessRegressor(kernel=k, n_restarts_optimizer=N_RESTARTS if restarts is None else restarts, random_state=SEED,
                                  optimizer=rec if record else "fmin_l_bfgs_b")
    gp.fit((Xs - mu_x) / sd_x, y - ysh - y0)
    gp.optimizer = "fmin_l_bfgs_b"                          # 保存（joblib）できるように戻す
    out = {"gp": gp, "mu_x": mu_x, "sd_x": sd_x, "y0": y0, "key": key, "cfg": dict(cfg), "th": float(th), "tv": float(tv), "cols": layout(key)[0], "mode": mode}
    if record:
        out["loss_runs"] = [list(h) for h in rec.runs]
    return out


def fit_job(key, tr, cfg, th, tv, restarts, theta0, mode, record=False):
    """fit_gp を並列に動かすための仕事（joblib から呼ぶ。設定は引数で渡す）"""
    warnings.simplefilter("ignore")
    return fit_gp(key, tr, cfg, th, tv, restarts, theta0, record, mode)


def predict(fit, df):
    """予測平均（ln η）、予測標準偏差（ノイズ込み）、潜在関数の標準偏差、ノイズの標準偏差"""
    _, Xs, ysh = design(df, fit["key"], fit["cfg"], fit["th"], fit["tv"], fit.get("mode", "shot"))
    mu, sd = fit["gp"].predict((Xs - fit["mu_x"]) / fit["sd_x"], return_std=True)
    noise = fit["gp"].kernel_.k2.noise_level
    return mu + fit["y0"] + ysh, sd, np.sqrt(np.maximum(sd ** 2 - noise, 0.0)), float(np.sqrt(noise))


def kernel_parts(gp):
    """学習したカーネルを、(Matern の部分〔振幅² × Matern〕, 線形の部分〔なければ None〕, ノイズ) に分ける"""
    k = gp.kernel_
    if isinstance(k.k1, Sum):
        return k.k1.k1, k.k1.k2, k.k2
    return k.k1, None, k.k2


def kernel_row(fit):
    """ハイパーパラメータ（振幅、長さスケール、線形項、ノイズ、対数周辺尤度、下限・上限への張り付き）"""
    gp = fit["gp"]
    prod, lin, white = kernel_parts(gp)
    ls = np.atleast_1d(prod.k2.length_scale)
    th_n, tv_n, _ = theta_names(fit["cfg"])
    row = {"model": fit["key"], th_n: fit["th"] if fit["key"].startswith("B") else np.nan, tv_n: fit["tv"] if fit["key"] != "C" else np.nan,
           "LML": float(gp.log_marginal_likelihood_value_), "amplitude": float(np.sqrt(prod.k1.constant_value)),
           "noise_var": float(white.noise_level), "noise_sd": float(np.sqrt(white.noise_level)),
           "noise_at_lower_bound": bool(white.noise_level < NOISE_LB * 1.05), "ls_at_bound": bool(np.any(ls > 99) or np.any(ls < 0.0105))}
    if lin is not None:
        row.update({"lin_amplitude": float(np.sqrt(lin.k1.constant_value)), "lin_sigma0": float(lin.k2.sigma_0)})
    row.update({f"ls_{c}": float(l) for c, l in zip(input_labels(fit["key"]), ls)})
    return row


# ---------------------------------------------------------------- プロファイル尤度（本体の手順5と同じ考え方）
def grid_of(lo, hi, step):
    return np.round(np.arange(lo, hi + step / 2.0, step), 6)


def fine_of(key, cfg):
    """細かい格子 (±幅, 刻み)：型の設定に "fine" があればそれ、なければ構造 A は FINE_A、B は FINE_B"""
    return cfg.get("fine") or (FINE_A if key.split("_")[0] == "A" else FINE_B)


def prof_row(f, th, tv):
    """格子の 1 点の結果（表の 1 行）。theta はその点で学習したカーネルの値（次の点や、最終モデルの学習の初期値に使う）"""
    return {"th": float(th), "tv": float(tv), "LML": float(f["gp"].log_marginal_likelihood_value_),
            "noise_sd": float(np.sqrt(f["gp"].kernel_.k2.noise_level)), "theta": f["gp"].kernel_.theta.tolist()}


def prof_chain(key, tr, cfg, points, restarts, mode, theta0=None):
    """格子の点 points = [(θ_h, θ_v), …] を順に学習して LML を返す。前の点の解を初期値にする（局所解でがたつくのを減らす）。
    最初の点は theta0 から（None なら、カーネルの既定の初期値から）"""
    warnings.simplefilter("ignore")
    out, theta = [], theta0
    for th, tv in points:
        f = fit_gp(key, tr, cfg, th, tv, restarts, theta, mode=mode)
        theta = f["gp"].kernel_.theta
        out.append(prof_row(f, th, tv))
    return out


def prof_points(key, tr, cfg, items, mode):
    """格子の点を 1 つずつ、決めた初期値から学習して LML を返す（見直しと、区間の延長に使う）。items = [(θ_h, θ_v, やり直しの回数, 初期値), …]"""
    warnings.simplefilter("ignore")
    return [prof_row(fit_gp(key, tr, cfg, th, tv, r, t0, mode=mode), th, tv) for th, tv, r, t0 in items]


def interval_gaps(pts, key, cfg):
    """LML が最大から 1.92 以内の点（おおよその 95% 区間）の隣（細かい格子の刻みだけ離れた点。粗い格子の範囲の中）のうち、まだ計算していない点。
    返り値：{(θ_h, θ_v): 隣の計算済みの点の theta}。空なら、区間の端が、計算した点の中で決まっている（区間の外側の隣の点まで計算済み）"""
    step = fine_of(key, cfg)[1]
    hg, vg = grid_of(*cfg["h_grid"]), grid_of(*cfg["v_grid"])
    is_b = key.split("_")[0] == "B"
    have = set(zip(np.round(pts["th"], 6), np.round(pts["tv"], 6)))
    dirs = [(0.0, step), (0.0, -step)] + ([(step, 0.0), (-step, 0.0)] if is_b else [])
    need = {}
    for r in pts[pts["LML"] >= pts["LML"].max() - 1.92].itertuples():
        for dh, dv in dirs:
            k = (round(float(r.th + dh), 6), round(float(r.tv + dv), 6))
            inside_range = (hg.min() - 1e-9 <= k[0] <= hg.max() + 1e-9 if is_b else k[0] == 0.0) and vg.min() - 1e-9 <= k[1] <= vg.max() + 1e-9
            if inside_range and k not in have and k not in need:
                need[k] = getattr(r, "theta", None)
    return need


def run_profile(key, tr, cfg, n_jobs=None, restarts=None, mode=None, extend=True):
    """θ の格子の各点で GPR を学習し、対数周辺尤度（LML）を求める（プロファイル尤度）。構造 A は θ_h = 0 の 1 次元、B は (θ_h, θ_v) の 2 次元。
    1. 粗い格子で全体を見る（θ_h ごとの列を、θ_v の小さい順にたどり、前の点の解を初期値にする。列を並列に計算する）。
    2. 最良の点を中心に、細かい格子を取る。新しい点は、θ_h ごとに、最良の点の θ_v から上と下へ（外へ向かって）たどり、最良の点の解から始める
       （2026-11-03。それまでは、列の端から既定の初期値で始めていたので、最初の点がノイズの大きい悪い解に入り、それが列の中で引き継がれることがあった）。
       最良の点が動いたら（細かい格子の端に当たったとき）、新しい最良の点を中心に取り直す（MAX_REFINE 回まで。斜めの尾根の上では、
       尤度がほとんど変わらないまま最良の点が動くことがある）。
    3. 見直し（2026-11-03）：すべての点を、最良の点の解を初期値にして最適化し直し（やり直しなし）、LML が上がった点は、その値に置き換える。
       最良の点が動いたら 2 へ戻る。
    4. 区間の延長（extend = True のとき。2026-11-03）：おおよその 95% 区間（最大から 1.92 以内）の点の隣に、まだ計算していない点があれば（interval_gaps）、
       隣の点の解から計算し、見直しもして、区間の端が細かい格子の中で決まるまで 1 列ずつ足す（MAX_EXTEND 回まで。粗い格子の範囲の外には出ない）。
       最良の点が動いたら 2 へ戻る。
    mode（既定 INPUT_MODE）は、並列の計算にも引数で渡す。
    返すもの：(表, 収束したか)。表の列：th, tv, LML, noise_sd, theta（カーネルの値。CSV には入れない）, stage（粗い／細かい／区間の延長）,
    review_gain（見直しで LML が上がった量。上がらなければ 0）。
    収束したか：最良の点の周り（±細かい格子の幅）の点がすべて計算済みで、見直しと区間の延長でも、最良の点が動かなかったか"""
    from joblib import Parallel, delayed
    n_jobs = N_JOBS if n_jobs is None else n_jobs
    restarts = PROFILE_RESTARTS if restarts is None else restarts
    mode = INPUT_MODE if mode is None else mode
    struct = key.split("_")[0]
    hg, vg = grid_of(*cfg["h_grid"]), grid_of(*cfg["v_grid"])
    rng, step = fine_of(key, cfg)
    P = {}                                                   # (θ_h, θ_v) → その点の結果（表の 1 行）

    def kp(th, tv):
        return (round(float(th), 6), round(float(tv), 6))

    def best():
        return max(P.values(), key=lambda r: r["LML"])

    def add(rows, stage):
        for r in rows:
            k = kp(r["th"], r["tv"])
            P[k] = {**r, "th": k[0], "tv": k[1], "stage": stage, "review_gain": 0.0}

    def chains(chs):
        """列 chs = [(点の並び, 最初の点の初期値), …] を並列に計算する"""
        res = Parallel(n_jobs=min(n_jobs, len(chs)))(delayed(prof_chain)(key, tr, cfg, ch, restarts, mode, t0) for ch, t0 in chs) if chs else []
        return [r for rr in res for r in rr]

    def points(items):
        """点を 1 つずつ（items = [(θ_h, θ_v, やり直しの回数, 初期値), …]）、並列に計算する"""
        n = min(n_jobs, len(items))
        res = Parallel(n_jobs=n)(delayed(prof_points)(key, tr, cfg, items[i::n], mode) for i in range(n)) if items else []
        return [r for rr in res for r in rr]

    def outward(b, new):
        """細かい格子の新しい点を、θ_h ごとに、最良の点の θ_v から上と下へ（外へ向かって）たどる列にする。どの列も、最良の点の解から始める"""
        by = {}
        for th, tv in new:
            by.setdefault((th, tv > b["tv"]), []).append((th, tv))
        t0 = np.asarray(b["theta"], float)
        return [(sorted(ps, key=lambda p: abs(p[1] - b["tv"])), t0) for _, ps in sorted(by.items())]

    def review(keys):
        """見直し：点 keys を、最良の点の解から最適化し直し（やり直しなし）、LML が上がった点を、その値に置き換える"""
        b = best()
        t0 = np.asarray(b["theta"], float)
        for r in points([(P[k]["th"], P[k]["tv"], 0, t0) for k in keys if k != kp(b["th"], b["tv"])]):
            q = P[kp(r["th"], r["tv"])]
            if r["LML"] > q["LML"] + 1e-3:                    # 計算の誤差ほどの差は、置き換えない
                q.update(LML=r["LML"], noise_sd=r["noise_sd"], theta=r["theta"], review_gain=q["review_gain"] + r["LML"] - q["LML"])

    def same(a, b):
        return kp(a["th"], a["tv"]) == kp(b["th"], b["tv"])

    add(chains([([(float(h), float(v)) for v in vg], None) for h in ([0.0] if struct == "A" else hg)]), "粗い")   # 1. 粗い格子
    refines, extends, converged = 0, 0, False
    while True:
        moved = True
        while refines < MAX_REFINE:                          # 2. 細かい格子（最良の点が動かなくなるまで）
            b = best()
            hs = [0.0] if struct == "A" else grid_of(b["th"] - rng, b["th"] + rng, step)
            new = [kp(h, v) for h in hs for v in grid_of(b["tv"] - rng, b["tv"] + rng, step) if kp(h, v) not in P]
            if new:
                add(chains(outward(b, new)), "細かい")
            refines += 1
            if same(best(), b):
                moved = False
                break
        if moved:                                            # 取り直しの回数の上限まで、最良の点が動き続けた（収束せず）
            break
        b = best()
        review(list(P))                                      # 3. 見直し
        if not same(best(), b):
            continue
        while extend and extends < MAX_EXTEND:               # 4. 区間の延長
            need = interval_gaps(pd.DataFrame(list(P.values())), key, cfg)
            if not need:
                break
            add(points([(h, v, restarts, np.asarray(t, float)) for (h, v), t in need.items()]), "区間の延長")
            review(list(need))
            extends += 1
            if not same(best(), b):
                break
        if not same(best(), b):
            continue
        converged = True
        break
    pts = pd.DataFrame(list(P.values())).sort_values(["th", "tv"]).reset_index(drop=True)
    return pts, converged


# 尤度比検定の目安（5% 基準）：θ = 0 との LML の差 ΔLML が、これ以上なら、足した係数の効果が認められる。
# 係数 1 つ（構造 A。B と A の比べ方にも使う）：2·ΔLML > 3.84（自由度 1 のカイ二乗分布の 95% 点）→ ΔLML > 1.92（95% 区間〔最大から 1.92 以内〕と同じ値）。
# 係数 2 つ（構造 B）：2·ΔLML > 5.99（自由度 2）→ ΔLML > 3.0
LRT_DLML_1 = 1.92
LRT_DLML = 3.0


def lrt_limit(key):
    """モデル key の、θ = 0 との LML の差 ΔLML の基準：構造 A（係数 1 つ）は LRT_DLML_1（1.92）、構造 B（係数 2 つ）は LRT_DLML（3.0）。
    2026-11-03 のユーザー決定（それまでは A にも 3.0 を使っていて、A の 95% 区間が 0 を含まないのに「支持されない」ことがあった）"""
    return LRT_DLML_1 if str(key).split("_")[0] == "A" else LRT_DLML


def profile_best(pts, key, cfg, converged=True, lml_zero=None):
    """最良の θ と、LML が最大から 1.92 以内の範囲（おおよその 95% 区間）。at_edge：最良の点が粗い格子の端。refine_converged：細かい格子の取り直しが収束したか。
    範囲は、計算した点の中だけで決まる（斜めの尾根の上では、実際の範囲はもっと長い）。尾根に沿った方向は決まりにくく、差 θ_v − θ_h（diff_*）が最もよく決まる。
    interval_closed：範囲の点の隣（細かい格子の刻み）が、すべて計算済み（範囲の端が、細かい格子の中で決まった）。interval_at_range：範囲が粗い格子の範囲の端に届いた
    （その外は計算しないので、範囲はもっと広いかもしれない）。n_review_up：見直しで LML が上がった点の数。review_gain_max：その上がり方の最大"""
    r = pts.loc[pts["LML"].idxmax()]
    inside = pts[pts["LML"] >= r["LML"] - 1.92]
    hg, vg = grid_of(*cfg["h_grid"]), grid_of(*cfg["v_grid"])
    z = pts[np.isclose(pts["th"], 0.0) & np.isclose(pts["tv"], 0.0)]
    lz = float(z["LML"].iloc[0]) if len(z) else (np.nan if lml_zero is None else lml_zero)
    diff = inside["tv"] - inside["th"]
    edge = bool(r["tv"] in (vg.min(), vg.max()) or (key.startswith("B") and r["th"] in (hg.min(), hg.max())))
    rim = bool(np.isclose(inside["tv"], vg.min()).any() or np.isclose(inside["tv"], vg.max()).any()
               or (key.startswith("B") and (np.isclose(inside["th"], hg.min()).any() or np.isclose(inside["th"], hg.max()).any())))
    gain = pts["review_gain"] if "review_gain" in pts.columns else pd.Series(0.0, index=pts.index)
    return {"th_best": float(r["th"]), "tv_best": float(r["tv"]), "diff_best": float(r["tv"] - r["th"]), "LML_best": float(r["LML"]),
            "LML_at_0": lz, "dLML_vs_0": float(r["LML"]) - lz, "th_lo": float(inside["th"].min()), "th_hi": float(inside["th"].max()),
            "tv_lo": float(inside["tv"].min()), "tv_hi": float(inside["tv"].max()), "diff_lo": float(diff.min()), "diff_hi": float(diff.max()),
            "n_in_95": int(len(inside)), "n_points": int(len(pts)), "at_edge": edge, "refine_converged": bool(converged),
            "interval_closed": not interval_gaps(pts, key, cfg), "interval_at_range": rim,
            "n_review_up": int((gain > 0).sum()), "review_gain_max": float(gain.max()) if len(gain) else 0.0,
            "theta_best": np.asarray(r["theta"], float) if "theta" in pts.columns else None,
            "theta_zero": np.asarray(z["theta"].iloc[0], float) if (len(z) and "theta" in pts.columns) else None}


def profile_careful(pts, b, key, cfg, fit_best, fit_zero):
    """念入りな最適化（最良の θ と θ = 0 の 2 点。fit_best・fit_zero）の LML を、格子の表 pts に入れ（格子の値と大きいほうを残す）、
    ΔLML と 95% 区間を、どちらもその表から出し直す（2026-11-03 のユーザー決定。それまでは、95% 区間は格子の値、ΔLML は念入りな最適化の値から出していて、
    θ = 0 が区間の外なのに ΔLML が 1.92 未満、のような食い違いが起きることがあった）。
    b：profile_best の結果（格子の値）。格子の値は LML_best_grid・LML_at_0_grid・dLML_vs_0_grid に残す。最良の点（th_best・tv_best）と theta_* は b のまま
    （最良の θ の学習が、そのまま最終モデルになるため）。表の列 careful は、念入りな最適化の値に置き換えた点。返り値：(新しい表, 新しい b)"""
    pts = pts.copy()
    pts["careful"] = pts["careful"] if "careful" in pts.columns else False
    lml = {}
    for name, (th, tv), f in [("best", (b["th_best"], b["tv_best"]), fit_best), ("zero", (0.0, 0.0), fit_zero)]:
        v, nz = float(f["gp"].log_marginal_likelihood_value_), float(np.sqrt(f["gp"].kernel_.k2.noise_level))
        m = np.isclose(pts["th"], th) & np.isclose(pts["tv"], tv)
        if not m.any():                                      # θ = 0 が格子にない（FORM_CFG の範囲に 0 を含めなかった）とき
            pts = pd.concat([pts, pd.DataFrame([{"th": float(th), "tv": float(tv), "LML": v, "noise_sd": nz, "stage": "念入り", "careful": True}])], ignore_index=True)
            m = np.isclose(pts["th"], th) & np.isclose(pts["tv"], tv)
        up = m & (pts["LML"] < v)
        pts.loc[up, "LML"], pts.loc[up, "noise_sd"], pts.loc[up, "careful"] = v, nz, True
        lml[name] = float(pts.loc[m, "LML"].max())
    nb = profile_best(pts, key, cfg, b["refine_converged"])
    out = {**b, **{k: nb[k] for k in ("th_lo", "th_hi", "tv_lo", "tv_hi", "diff_lo", "diff_hi", "n_in_95", "n_points", "interval_closed", "interval_at_range")}}
    out.update(LML_best_grid=b["LML_best"], LML_at_0_grid=b["LML_at_0"], dLML_vs_0_grid=b["dLML_vs_0"],
               LML_best=lml["best"], LML_at_0=lml["zero"], dLML_vs_0=lml["best"] - lml["zero"])
    return pts, out


def shift_candidates(rows):
    """シフトファクター検討用のノートのまとめで、1 つの材料の、学習用のノートの SHIFT_COL・SHIFT_STRUCT・SHIFT_FORM の候補を選ぶ（2026-11-03 のユーザー決定）。
    rows：シフトのあるモデルごとの {"モデル", "LML", "dLML_vs_0"}（LML・ΔLML は念入りな最適化の値）。
    1. 列（Tset・Tff）と型（Arr・WLF）の組ごとに：A が θ = 0 より基準 lrt_limit（1.92）以上大きければ、その組の温度シフトは支持される。
       支持されたら、B が A より LRT_DLML_1（1.92）以上大きいときだけ B、そうでなければ A（B は A に横の係数を 1 つ足したもの。B − A が負になるのは、
       B の格子が A より粗いためで、0 と同じ扱い）。A が支持されなくても、B が θ = 0 より LRT_DLML（3.0）以上大きければ、B が A より 1.92 以上大きいときは B、
       そうでなければ ★（どちらとも決まらない）。A・B の片方だけを学習したときは、その片方だけで決める。
    2. 残った組どうしは、LML の大きいものを選ぶ（θ = 0 のモデルが列ごとに違うので、ΔLML ではなく LML で比べる）。係数の数が違う（A と B の）組が残ったら ★。
    3. どの組も残らなければ、None（シフトなしの C）。
    返り値：(候補のモデルの記号〔None は C〕, 組ごとの表, 注意〔★ の理由。なければ空のリスト〕)"""
    by = {}
    for r in rows:
        s, t = str(r["モデル"]).split("_")[:2]
        by.setdefault((t, key_form(r["モデル"])), {})[s] = r
    tab, keep, notes = [], [], []
    for (t, form), g in by.items():
        a, b = g.get("A"), g.get("B")
        a_ok = a is not None and a["dLML_vs_0"] >= lrt_limit(a["モデル"])
        b_ok = b is not None and b["dLML_vs_0"] >= lrt_limit(b["モデル"])
        gain = (b["LML"] - a["LML"]) if (a is not None and b is not None) else np.nan
        if a is not None and b is not None:
            pick = (b if gain >= LRT_DLML_1 else a) if a_ok else ((b if gain >= LRT_DLML_1 else "★") if b_ok else None)
        else:
            pick = (a if a_ok else None) if a is not None else (b if b_ok else None)
        name = f"{t}・{FORM_JA.get(form, form)}"
        if pick == "★":
            notes.append(f"{name}：B は θ = 0 より {b['dLML_vs_0']:.2f} 大きい（基準 {LRT_DLML:.1f} 以上）が、A は {a['dLML_vs_0']:.2f}（基準 {LRT_DLML_1:g} 未満）で、"
                         f"B と A の差 {gain:.2f} も {LRT_DLML_1:g} 未満なので、A か B かが決まらない（この組は候補から外した）")
        elif pick is not None:
            keep.append(pick)
        tab.append({"シフトに使う温度": t, "型": FORM_JA.get(form, form),
                    "A の ΔLML": a["dLML_vs_0"] if a is not None else np.nan, f"A は支持（{LRT_DLML_1:g} 以上）": a_ok if a is not None else None,
                    "B の ΔLML": b["dLML_vs_0"] if b is not None else np.nan, f"B は支持（{LRT_DLML:.1f} 以上）": b_ok if b is not None else None,
                    "B − A の LML": gain, "この組の結論": "決まらない（★）" if pick == "★" else (pick["モデル"] if pick is not None else "支持されない")})
    if not keep:
        return None, pd.DataFrame(tab), notes
    win = max(keep, key=lambda r: r["LML"])
    n_coef = {str(r["モデル"]).split("_")[0] for r in keep}
    if len(n_coef) > 1:
        notes.append("係数の数が違うモデル（" + "、".join(f"{r['モデル']} LML {r['LML']:.1f}" for r in keep)
                     + f"）が残った。LML の最も大きい {win['モデル']} を候補にしたが、係数の多い B は LML が大きく出やすいので、確かめる")
    return win["モデル"], pd.DataFrame(tab), notes


# ---------------------------------------------------------------- 評価
def evaluate(y, mu, sd):
    """ln η の指標：R2、RMSE、95% 予測区間の被覆率、NLPD、予測標準偏差の平均。R2_real は粘度の実数値での R2（6月のノートと同じ）"""
    z = (y - mu) / sd
    return {"R2": r2_score(y, mu) if len(y) > 1 else np.nan, "RMSE": float(np.sqrt(mean_squared_error(y, mu))),
            "coverage95": float(np.mean(np.abs(z) <= 1.96)), "NLPD": float(np.mean(0.5 * np.log(2 * np.pi * sd ** 2) + 0.5 * z ** 2)),
            "mean_sd": float(np.mean(sd)), "R2_real": r2_score(np.exp(y), np.exp(mu)) if len(y) > 1 else np.nan}


def metrics_rows(mat, key, te, mu, sd, sd_f, extra=None):
    """検証の指標を、全体・低速側・高速側（と、感度確認の条件を除いたもの）ごとに返す"""
    subsets = {"全体": np.ones(len(te), bool)}
    for g in ["低速側", "高速側", "範囲内"]:
        if (te["group"] == g).any():
            subsets[g] = (te["group"] == g).to_numpy()
    if te["is_sens"].any():
        subsets.update({s + "（感度確認を除く）": m & ~te["is_sens"].to_numpy() for s, m in list(subsets.items())})
    rows = []
    y = te["ln_eta"].to_numpy()
    for s, m in subsets.items():
        if m.sum() == 0:
            continue
        rows.append({"材料": MAT_NAMES[mat], "モデル": key, "範囲": s, "行数": int(m.sum()), "条件数": int(te.loc[m, "cond"].nunique()),
                     **evaluate(y[m], mu[m], sd[m]), "mean_sd_latent": float(np.nanmean(sd_f[m])) if np.isfinite(sd_f[m]).any() else np.nan, **(extra or {})})
    return rows


def judge(ok, text, mat=None, section=""):
    """結果の判定を 1 行で表示し、記録する。ok が True なら「OK」、False なら「★要確認」"""
    head = f"[{MAT_NAMES[mat]}] " if mat else ""
    print(f">> 判定：{'OK' if ok else '★要確認'}　{head}{text}")
    JUDGES.append({"節": section, "材料": MAT_NAMES[mat] if mat else "", "判定": "OK" if ok else "★要確認", "内容": text})


def to_csv(df, path):
    df.to_csv(path, index=False, encoding="utf-8-sig")


# ---------------------------------------------------------------- 外れ値の検知（本体の check_data.py と同じ規則。05 のデータ用）
def cond_means(df):
    """条件ごとの平均（ln η と ln γ̇ をショットで平均した点）"""
    g = df.groupby("cond", sort=False)
    t = g.agg(Tset=("Tset", "first"), H=("H", "first"), v=("v", "first"), ln_eta=("ln_eta", "mean"), ln_gd=("ln_gd", "mean"),
              n=("ln_eta", "size"), grp=("grp", "first")).reset_index()
    t["ln_H"] = np.log(t["H"])
    return t


def outlier_scan(df):
    """規則 f：条件平均が、ほかの条件から予測した値と大きく違う条件を拾う（本体の nb_common.outlier_scan と同じ）。
    条件平均の GPR（入力 ln γ̇（条件平均）・ln H・Tset、線形項あり）の、条件を 1 つずつ抜いた予測（閉じた式）との残差を、予測標準偏差で割って z とする。
    ※ この検知だけは、データの確認のための道具なので、本体と同じく H を使う（モデルの説明変数ではない）"""
    t = cond_means(df)
    X, y = t[["ln_gd", "ln_H", "Tset"]].to_numpy(), t["ln_eta"].to_numpy()
    Xs, yc = StandardScaler().fit_transform(X), y - y.mean()
    gp = GaussianProcessRegressor(kernel=kernel_for(3, True), n_restarts_optimizer=N_RESTARTS, random_state=SEED).fit(Xs, yc)
    Kinv = np.linalg.inv(gp.kernel_(Xs))
    d = np.diag(Kinv)
    t["resid"], t["sd"] = (Kinv @ yc) / d, np.sqrt(1.0 / d)       # 条件 i を抜いた予測：平均 = y_i − (K⁻¹y)_i / K⁻¹_ii、分散 = 1 / K⁻¹_ii
    t["z"] = t["resid"] / t["sd"]
    t["flag"] = (t["z"].abs() > OUTLIER_Z) & (t["resid"].abs() > OUTLIER_RESID)
    return t[["cond", "grp", "Tset", "H", "v", "n", "resid", "sd", "z", "flag"]]


def registered(mat, grp, row, plist):
    """一覧の「除外」「感度確認」に、この条件（row に shot があれば、そのショット）が登録されているか（ID を返す）"""
    one = pd.DataFrame({"Tset": [float(row["Tset"])], "H": [float(row["H"])], "v": [float(row["v"])], "shot": [row.get("shot", -1)]})
    if pd.isna(one["shot"].iloc[0]):
        one["shot"] = -1
    _, ids = problem_mask(one, mat, grp, plist, ("除外", "感度確認"))
    return ids[0]


def candidates(mat, pool, plist):
    """問題の候補（見直しの候補であって、外れ値と判定したものではない。一覧に載せるかはユーザーが決める）。
    a 温度の間で γ̇ が 1.3 倍を超えて違う／b 最低温度の粘度が最高温度より小さい（順序が逆）／c 同じ条件内の ln η の標準偏差 > 0.1／
    e 別のショットと値が完全に同じ行／g P3_3 が条件の中央値から P33_DEV 以上離れたショット"""
    cand = []
    g = pool.groupby(["H", "v", "Tset"]).agg(gd=("gd", "mean"), eta=("eta", "mean"), grp=("grp", "first")).reset_index()
    t_lo, t_hi = pool["Tset"].min(), pool["Tset"].max()
    for (h, v), s in g.groupby(["H", "v"]):
        if len(s) < 3:
            continue
        lg = np.log(s["gd"].to_numpy())
        if np.exp(lg.max() - lg.min()) > 1.3:
            odd = s.iloc[np.argmax(np.abs(lg - np.median(lg)))]
            cand.append({"規則": "a 温度の間でせん断速度が1.3倍超違う", "組": odd["grp"], "設定温度": int(odd["Tset"]), "厚み": h, "射出速度": int(v), "ショット": "",
                         "内容": f"γ̇ の最大/最小 = {np.exp(lg.max() - lg.min()):.2f}"})
        e = s.set_index("Tset")["eta"]
        if t_lo in e.index and t_hi in e.index and e[t_lo] / e[t_hi] < 0.95:
            cand.append({"規則": "b 粘度の順序が温度と逆", "組": s["grp"].iloc[0], "設定温度": int(t_lo), "厚み": h, "射出速度": int(v), "ショット": "",
                         "内容": f"η({int(t_lo)})/η({int(t_hi)}) = {e[t_lo] / e[t_hi]:.2f}"})
    sd = pool.groupby(["Tset", "H", "v"]).agg(std=("ln_eta", "std"), size=("ln_eta", "size"), grp=("grp", "first")).reset_index()
    for _, r in sd[(sd["size"] >= 2) & (sd["std"] > 0.1)].iterrows():
        cand.append({"規則": "c 同じ条件内のばらつきが大きい", "組": r["grp"], "設定温度": int(r["Tset"]), "厚み": r["H"], "射出速度": int(r["v"]), "ショット": "",
                     "内容": f"ln η の標準偏差 = {r['std']:.3f}（{int(r['size'])} ショット）"})
    key = [c for c in ["eta", "gd", "P31", "P32", "P33"] if c in pool.columns and pool[c].notna().all()]
    for grp, df in pool.groupby("grp"):
        dup = df[df.duplicated(key, keep=False)]
        for _, gg in dup.groupby(key):
            if gg["file"].nunique() < 2:
                continue
            r0 = gg.sort_values("file").iloc[0]
            cand.append({"規則": "e 別のショットと同じ値の行", "組": grp, "設定温度": int(r0["Tset"]), "厚み": r0["H"], "射出速度": int(r0["v"]), "ショット": str(int(r0["shot"])),
                         "内容": f"{' と '.join(gg['file'])} の {'・'.join(key)} が完全に一致"})
    med = pool.groupby("cond")["P33"].transform("median")
    n = pool.groupby("cond")["P33"].transform("size")
    dev = (pool["P33"] / med - 1.0).abs()
    for _, r in pool[(n >= 3) & (dev >= P33_DEV)].iterrows():
        cand.append({"規則": f"g P3_3 が条件の中央値から{int(P33_DEV * 100)}%以上離れている", "組": r["grp"], "設定温度": int(r["Tset"]), "厚み": r["H"], "射出速度": int(r["v"]),
                     "ショット": str(int(r["shot"])), "内容": f"P3_3 = {r['P33']:.2f} MPa（条件の中央値 {med[r.name]:.2f}、{(r['P33'] / med[r.name] - 1) * 100:+.0f}%）"})
    out = pd.DataFrame(cand, columns=["規則", "組", "設定温度", "厚み", "射出速度", "ショット", "内容"])
    out.insert(0, "材料", MAT_NAMES[mat])
    out["一覧の ID"] = [registered(mat, r["組"], {"Tset": r["設定温度"], "H": r["厚み"], "v": r["射出速度"],
                                                   "shot": int(r["ショット"]) if r["ショット"] != "" else np.nan}, plist) for _, r in out.iterrows()]
    return out


# ---------------------------------------------------------------- Cross-WLF（6月のノートのセル10.5・11。D3 = 0）
CWLF_NAMES = ["n", "tau_star", "D1", "D2", "A1", "A2"]


def cross_wlf_ln(T_c, gd, n, ln_tau, ln_D1, D2, A1, A2):
    """ln η（Cross-WLF。D3 = 0 なので T* = D2）。数値の安定のため、τ* と D1 は ln で受け取る。T は ℃、D2 は K"""
    diff = np.maximum(np.asarray(T_c, float) + 273.15 - D2, 1.0)
    ln_eta0 = ln_D1 - A1 * diff / (A2 + diff)
    return ln_eta0 - np.logaddexp(0.0, (1.0 - n) * (ln_eta0 + np.log(gd) - ln_tau))


CWLF_STARTS = globals().get("CWLF_STARTS", 12)            # Cross-WLF の当てはめの初期値の数（局所解を避けるため、乱数の初期値を何度か試して、残差が最小のものを採る）


def fit_cross_wlf(tr, t_col, fixed):
    """Cross-WLF の係数を、学習データ（ln η の最小二乗）で当てはめる。fixed：{名前: 値 または None（探索）}。
    初期値は、6月のノートに近い値と、探索範囲の中の乱数（SEED で固定）の CWLF_STARTS 個で、残差が最小のものを採る。
    返す辞書：係数（実数）、学習データでの残差の標準偏差"""
    from scipy.optimize import curve_fit
    init = {"n": 0.3, "tau_star": np.log(3e4), "D1": np.log(1e12), "D2": 263.15, "A1": 25.0, "A2": 51.6}
    lo = {"n": 0.0, "tau_star": np.log(1e2), "D1": np.log(1e3), "D2": 150.0, "A1": 0.5, "A2": 10.0}
    hi = {"n": 0.95, "tau_star": np.log(1e7), "D1": np.log(1e30), "D2": 450.0, "A1": 80.0, "A2": 200.0}
    conv = {k: (np.log(v) if k in ("tau_star", "D1") and v is not None else v) for k, v in fixed.items()}
    free = [k for k in CWLF_NAMES if conv.get(k) is None]

    def full(args):
        vals = dict(conv)
        vals.update(dict(zip(free, args)))
        return [vals[k] for k in CWLF_NAMES]

    T, gd, y = tr[t_col].to_numpy(float), tr["gd"].to_numpy(float), tr["ln_eta"].to_numpy()
    f = lambda X, *args: cross_wlf_ln(X[0], X[1], *full(args))
    rs = np.random.RandomState(SEED)
    starts = [[init[k] for k in free]] + [[rs.uniform(lo[k], hi[k]) for k in free] for _ in range(max(CWLF_STARTS - 1, 0))]
    best = None
    for p0 in starts:
        try:
            p, _ = curve_fit(f, np.vstack([T, gd]), y, p0=p0, bounds=([lo[k] for k in free], [hi[k] for k in free]), maxfev=20000)
        except Exception:
            continue
        sse = float(np.sum((y - f(np.vstack([T, gd]), *p)) ** 2))
        if best is None or sse < best[0]:
            best = (sse, p)
    if best is None:
        raise RuntimeError("Cross-WLF の当てはめがすべての初期値で失敗しました")
    popt = best[1]
    vals = full(popt)
    resid = y - cross_wlf_ln(T, gd, *vals)
    at_bound = [k for k, v in zip(free, popt) if np.isclose(v, lo[k], rtol=0, atol=1e-6 * max(1.0, abs(lo[k]))) or np.isclose(v, hi[k], rtol=0, atol=1e-6 * max(1.0, abs(hi[k])))]
    out = dict(zip(CWLF_NAMES, vals))
    out["tau_star"], out["D1"] = float(np.exp(out["tau_star"])), float(np.exp(out["D1"]))
    return {"params": out, "free": free, "at_bound": at_bound, "resid_sd": float(np.sqrt(np.mean(resid ** 2))), "t_col": t_col}


def predict_cross_wlf(cw, df):
    p = cw["params"]
    mu = cross_wlf_ln(df[cw["t_col"]].to_numpy(float), df["gd"].to_numpy(float), p["n"], np.log(p["tau_star"]), np.log(p["D1"]), p["D2"], p["A1"], p["A2"])
    return mu, np.full(len(df), cw["resid_sd"])


# ---------------------------------------------------------------- 予測の入力を組み立てる（依存性グラフ・3次元グラフ・保存したモデルの使い方）
VAR_JA = {"gd": ("せん断速度 γ̇", "(1/s)", True), "P33": ("流動中樹脂圧力 P3_3", "(MPa)", True),
          "Tff": ("流動中樹脂温度 Tff", "(℃)", False), "Tset": ("加熱シリンダ温度 Tset", "(℃)", False),
          "Tboth": ("Tset と Tff を同時に", "(℃。横軸は Tset)", False)}     # (名前, 単位, 対数軸か)。Tboth：設定温度を上げると、流動中樹脂温度も同じだけ上がるとして、2 つを一緒に動かす


def model_inputs(gd, P33, Tff, Tset):
    """説明変数の実数値（γ̇ [1/s]、P3_3 [MPa]、Tff [℃]、Tset [℃]。配列でも数でもよい）→ predict に渡せる表"""
    gd, P33, Tff, Tset = (np.asarray(a, float).ravel() for a in np.broadcast_arrays(gd, P33, Tff, Tset))
    return pd.DataFrame({"gd": gd, "ln_gd": np.log(gd), "P33": P33, "ln_P33": np.log(P33 * 1e6), "Tff": Tff, "ln_Tff": np.log(Tff + 273.15), "Tset": Tset})


def base_point(tr, cfg):
    """依存性グラフで固定する値：学習データの中央値（γ̇・P3_3 は対数の中央値）。設定温度は基準温度 Tref"""
    return {"gd": float(np.exp(tr["ln_gd"].median())), "P33": float(np.exp(tr["ln_P33"].median()) / 1e6),
            "Tff": float(np.exp(tr["ln_Tff"].median()) - 273.15), "Tset": float(cfg["Tref"])}


def var_grid(pool, var, n, pad=0.1):
    """動かす説明変数の格子（データの範囲を少し広げる。対数軸の変数は対数で等間隔。設定温度は ±10 ℃）"""
    if var in ("Tset", "Tboth"):
        return np.linspace(pool["Tset"].min() - 10.0, pool["Tset"].max() + 10.0, n)
    x = pool[var].to_numpy(float)
    if VAR_JA[var][2]:
        lo, hi = np.log(x.min()), np.log(x.max())
        return np.exp(np.linspace(lo - pad * (hi - lo), hi + pad * (hi - lo), n))
    lo, hi = x.min(), x.max()
    return np.linspace(lo - pad * (hi - lo), hi + pad * (hi - lo), n)


def superpose_rmse(t):
    """重ね合わせの目安：厚みごとに、条件平均の点（x = ln γ̇〔換算〕、y = ln η〔換算〕）を 3 次式で当てはめた残差の RMSE（全温度をまとめて）。
    温度のシフトで曲線が重なるほど小さくなる"""
    res = []
    for _, g in t.groupby("H"):
        if len(g) < 6:
            continue
        c = np.polyfit(g["x"], g["y"], 3)
        res += list(g["y"] - np.polyval(c, g["x"]))
    return float(np.sqrt(np.mean(np.square(res)))) if res else np.nan


def shifted_cond_means(df, key, cfg, th, tv):
    """条件平均の、換算した ln γ̇（x）と ln η（y）。key が None ならシフト前。Tff 版はショットごとの Tff でずらしてから平均する"""
    x, y = df["ln_gd"].to_numpy(), df["ln_eta"].to_numpy()
    if key is not None:
        T = df[layout(key)[1]].to_numpy(float)
        x = x + shift_ln(T, th, cfg)
        y = y - shift_ln(T, tv, cfg)
    return df.assign(x=x, y=y).groupby("cond").agg(x=("x", "mean"), y=("y", "mean"), Tset=("Tset", "first"), H=("H", "first")).reset_index()


# ---------------------------------------------------------------- 図の保存と、色・記号
H_COLORS = {0.5: "#d62728", 1.0: "#1f77b4", 1.5: "#f0b400", 2.0: "#2ca02c"}       # 厚みごとの色（6月のノートと同じ並び）
H_MARKS = {0.5: "o", 1.0: "s", 1.5: "^", 2.0: "D"}
T_PALETTE = ["#1f77b4", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]               # 設定温度ごとの色（低い順）
T_MARKS = ["o", "s", "^", "D", "v"]


def t_style(levels):
    """設定温度の水準 → (色, 記号) の辞書（低い順に青・緑・赤）"""
    lv = sorted(levels)
    return {t: (T_PALETTE[i % len(T_PALETTE)], T_MARKS[i % len(T_MARKS)]) for i, t in enumerate(lv)}


def log_ticks(ax, axis="xy"):
    """対数軸の目盛りを、数字が重ならない形にする（軸の範囲を決めた後で呼ぶ）。範囲が 2.5 桁以下なら、1・2・5 × 10^n を普通の数で、
    それより広ければ、10 の累乗だけ。小さい目盛りには数字を付けない"""
    import matplotlib.ticker as ticker
    for name in axis:
        a = ax.xaxis if name == "x" else ax.yaxis
        lo, hi = ax.get_xlim() if name == "x" else ax.get_ylim()
        if hi > lo > 0 and np.log10(hi / lo) <= 2.5:
            a.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0, 2.0, 5.0)))
            a.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:g}"))
        else:
            a.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
            a.set_major_formatter(ticker.LogFormatterSciNotation())
        a.set_minor_formatter(ticker.NullFormatter())


def tick_labels_overlap(fig, axis, gap=0.4):
    """軸 axis の大きい目盛りの数字のうち、グラフの範囲の中のものが、隣どうしで重なるか（間が、数字の高さの gap 倍より狭いか）。図を一度描いて確かめる"""
    fig.canvas.draw()
    x = axis.axis_name == "x"
    ab = axis.axes.get_window_extent()
    lo, hi = (ab.x0, ab.x1) if x else (ab.y0, ab.y1)
    boxes = [t.get_window_extent() for t in axis.get_majorticklabels() if t.get_text()]
    spans = sorted((b.x0, b.x1) if x else (b.y0, b.y1) for b in boxes)
    spans = [s for s in spans if lo - 1 <= (s[0] + s[1]) / 2 <= hi + 1]
    pad = gap * max((b.height for b in boxes), default=0.0)
    return any(a[1] + pad > b[0] for a, b in zip(spans, spans[1:]))


def save_fig(fig, out_dir, stem, show=None):
    """図を out_dir に PNG で保存する（解像度 FIG_DPI。既定 600 dpi の 1 枚だけ。報告書・発表資料にそのまま使える。2026-10-27 から、低い解像度の版は作らない）。
    show（既定 SHOW_FIGS）なら表示し、閉じる。保存したファイルの一覧を返す"""
    import matplotlib.pyplot as plt
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"{stem}.png")
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    if SHOW_FIGS if show is None else show:
        plt.show()
    plt.close(fig)
    return [path]


DEP92_STYLE = {0.5: ("#d62728", "o", 14), 1.0: ("#1f77b4", "s", 13), 1.5: ("#ffd30ed9", "^", 12), 2.0: ("#41ea4f", "D", 11)}      # 6月のノートのセル9-2 の style_dict（色・形・zorder）
DEP92_VARS = [("gd", "せん断速度", r"$\gamma$", "(1/s)", True, "Shear_Rate"), ("P33", "流動中樹脂圧力", r"$P_{3\_3}$", "(MPa)", True, "P33"),
              ("Tff", "流動中樹脂温度", r"$T_{ff}$", "(℃)", False, "Tff"), ("Tset", "加熱シリンダ温度", r"$T_{set}$", "(℃)", False, "Tset")]      # (変数, 名前, 記号, 単位, 対数軸か, ファイル名)


def dependency_plots_92(fit, mat, tr, out_dir, stem="依存性グラフ", **kw):
    """GPR6月ノートのセル9-2（目的変数に対する各説明変数の依存性グラフ）と同じ図を、学習データ（tr）と学習済みモデルで作る。
    説明変数ごとに 1 枚：その変数だけを動かし、ほかは学習データの平均（対数の変数は、対数の平均）に固定した、予測平均（赤線）と 95% 区間（帯）に、
    学習データ全部の点（色・形は厚み）を重ねる。図の大きさ・文字・点・目盛り（対数軸は 10 の累乗）・凡例・タイトルは、6月のセル9-2 と同じ。
    設定（kw）：figsize、dpi（保存の解像度。None なら FIG_DPI）、axis_fs・legend_fs・title_fs・tick_fs、plot_marker・legend_marker、show_xlabel・show_ylabel、ylim、
    xlim（変数 → (下限, 上限)）、n_grid、formats、show。保存したファイルの一覧を返す"""
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    cfg = dict(figsize=(12, 10), dpi=None, axis_fs=24, legend_fs=24, title_fs=24, tick_fs=28, plot_marker=300, legend_marker=300, show_xlabel=False, show_ylabel=False,
               ylim=(0.1, 10000.0), xlim={"gd": (10.0, 100000.0)}, n_grid=100, formats=["png"], show=None)
    cfg.update(kw)
    dpi = cfg["dpi"] or FIG_DPI
    base = {"gd": float(np.exp(tr["ln_gd"].mean())), "P33": float(np.exp(tr["ln_P33"].mean()) / 1e6), "Tff": float(np.exp(tr["ln_Tff"].mean()) - 273.15), "Tset": float(tr["Tset"].mean())}
    saved = []
    for var, name, sym, unit, logx, fname in DEP92_VARS:
        fig, ax = plt.subplots(figsize=cfg["figsize"])
        x = tr[var].to_numpy(float)
        lim = cfg["xlim"].get(var)
        if lim is None:                                           # 6月と同じ：データの範囲を、両側に 10% 広げる（対数軸の変数は、対数で）
            lo, hi = (np.log(x.min()), np.log(x.max())) if logx else (x.min(), x.max())
            lo, hi = lo - 0.1 * (hi - lo), hi + 0.1 * (hi - lo)
            grid = np.exp(np.linspace(lo, hi, cfg["n_grid"])) if logx else np.linspace(lo, hi, cfg["n_grid"])
        else:
            grid = np.exp(np.linspace(np.log(lim[0]), np.log(lim[1]), cfg["n_grid"])) if logx else np.linspace(lim[0], lim[1], cfg["n_grid"])
        mu, sd, _, _ = predict(fit, model_inputs(**{**base, var: grid}))
        lower, upper = np.exp(mu - 1.96 * sd), np.exp(mu + 1.96 * sd)
        line = ax.plot(grid, np.exp(mu), color="red", linewidth=2, label="予測平均値", zorder=20)
        fill = ax.fill_between(grid, lower, upper, color="red", alpha=0.25, label="95% 信頼区間", zorder=15)
        ax.plot(grid, lower, color="red", linewidth=0.5, alpha=0.6, zorder=16)
        ax.plot(grid, upper, color="red", linewidth=0.5, alpha=0.6, zorder=16)
        hs, labs = [], []
        for h in sorted(tr["H"].unique()):
            m = np.isclose(tr["H"].to_numpy(), h)
            col, mk, z = DEP92_STYLE.get(float(h), ("gray", "x", 10))
            sc = ax.scatter(x[m], tr.loc[m, "eta"], color=col, s=cfg["plot_marker"], marker=mk, alpha=0.8, edgecolors="k", linewidths=0.5, zorder=z)
            hs.append(sc)
            labs.append(f"{h:g} mm")
        scale = np.sqrt(cfg["legend_marker"] / cfg["plot_marker"])
        ax.legend([line[0], fill, Patch(color="none", label="")] + hs, ["予測平均値", "95% 信頼区間", "キャビティ厚み"] + labs, loc="upper right", fontsize=cfg["legend_fs"],
                  framealpha=1.0, edgecolor="black", markerscale=scale)
        ax.tick_params(axis="both", which="both", direction="in", top=True, right=True, labelsize=cfg["tick_fs"], pad=8)
        ax.set_title("ガウス過程回帰による予測モデル\n（せん断速度と粘度の関係）" if var == "gd" else f"樹脂粘度の {name} 依存性", fontsize=cfg["title_fs"], fontweight="bold", pad=15)
        if cfg["show_xlabel"]:
            ax.set_xlabel(f"{name} {sym} {unit}", fontsize=cfg["axis_fs"], fontweight="bold", labelpad=15)
        if cfg["show_ylabel"]:
            ax.set_ylabel("樹脂粘度 (Pa·s)", fontsize=cfg["axis_fs"], fontweight="bold", labelpad=15)
        ax.set_yscale("log")
        if cfg["ylim"] is not None:
            ax.set_ylim(*cfg["ylim"])
        ax.yaxis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
        ax.yaxis.set_major_formatter(ticker.LogFormatterSciNotation())
        if lim is not None:
            ax.set_xlim(*lim)
        if logx:
            ax.set_xscale("log")
            ax.xaxis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
            ax.xaxis.set_major_formatter(ticker.LogFormatterSciNotation())
            ax.xaxis.set_minor_formatter(ticker.NullFormatter())
            ax.tick_params(axis="x", which="both", direction="in", top=True, labelsize=cfg["tick_fs"], pad=8)
        ax.grid(True, which="major", linestyle="-", alpha=0.6, color="gray", zorder=1)
        ax.grid(True, which="minor", linestyle=":", alpha=0.4, zorder=1)
        fig.tight_layout()
        os.makedirs(out_dir, exist_ok=True)
        for fmt in cfg["formats"]:
            path = os.path.join(out_dir, f"{stem}_{fname}.{fmt}")
            fig.savefig(path, bbox_inches="tight", dpi=dpi)
            saved.append(path)
        if SHOW_FIGS if cfg["show"] is None else cfg["show"]:
            plt.show()
        plt.close(fig)
    return saved


def flow_inputs(sub, lg, h, t):
    """予測線の入力（ln γ̇ = lg の点）。P3_3・Tff は、同じ設定温度・厚みの条件平均を ln γ̇ で内挿する（範囲の外は端の値のまま）"""
    cm = sub.groupby("cond").agg(lg=("ln_gd", "mean"), p=("ln_P33", "mean"), tf=("ln_Tff", "mean")).sort_values("lg")
    ip = lambda c: np.interp(lg, cm["lg"], cm[c])
    ln_tff = ip("tf")
    return pd.DataFrame({"ln_gd": lg, "gd": np.exp(lg), "ln_P33": ip("p"), "ln_Tff": ln_tff, "Tff": np.exp(ln_tff) - 273.15, "Tset": float(t), "H": float(h)})


FLOW_DEFAULT = dict(layouts=["temperature", "thickness"], band="predict", points="cond", show_test=True, mark_sens=True, x_range=None, extend=1.3,
                    xlim={"thickness": None, "temperature": None}, ylim={"thickness": None, "temperature": None}, figsize=(12, 10), dpi=None,
                    font={"axis": 24, "legend": 24, "title": 24, "tick": 28}, marker=300, marker_shot=70, line=2.0, show_title=True,
                    show_xlabel=False, show_ylabel=False, formats=["png"], n_grid=120, show=None,
                    legend_separate=False, point_color=True, point_edge=1.5, point_fill="white", point_fill_alpha=1.0)
# legend_separate：True なら、グラフ本体に凡例を付けず、凡例だけを別の画像（<図の名前>_凡例）に保存する（2026-11-08 追加。既定は今までどおり）
# point_color：False なら、データ点を色なし（塗りつぶしの色 point_fill〔既定は白〕と黒い枠線。検証〔3data〕の点は灰色の枠線）にする。枠線の太さは point_edge（2026-11-08 追加。同日、ユーザーの依頼で、塗りつぶしなし → 白の塗りつぶしに変えた）
# point_fill・point_fill_alpha：色なしのときの、点の塗りつぶしの色と濃さ（0〜1。1 で不透明。枠線は不透明のまま。2026-11-09 のユーザーの依頼で追加）
POINT_TEST_EDGE = "0.55"            # 色なしのときの、検証（3data）の点の枠線の色（灰色）


def fill_rgba(color, alpha):
    """塗りつぶしの色（名前・"#rrggbb"・"0.8" などの灰色）と濃さ alpha を、RGBA の組にする（枠線の濃さは変えないため、塗りつぶしだけに濃さを付ける）"""
    from matplotlib.colors import to_rgba
    return to_rgba(color, alpha)


def point_kw(cfg, col, test=False):
    """データ点の scatter の引数（色・枠）。cfg["point_color"] が True なら今までどおりの色付き、False なら塗りつぶし（色 point_fill・濃さ point_fill_alpha）と枠線（学習は黒、検証は灰色。太さ point_edge）"""
    if not cfg["point_color"]:
        return dict(facecolors=[fill_rgba(cfg["point_fill"], cfg["point_fill_alpha"])], edgecolors=POINT_TEST_EDGE if test else "k", linewidths=cfg["point_edge"])
    if test:
        return dict(facecolors="white", edgecolors=col, linewidths=2.5)
    return dict(color=col, alpha=0.85, edgecolors="k", linewidths=0.6)


def point_handle_kw(cfg, col, test=False, edge=0.6):
    """凡例の点の記号の引数（point_kw と同じ見た目）"""
    if not cfg["point_color"]:
        return dict(markerfacecolor=fill_rgba(cfg["point_fill"], cfg["point_fill_alpha"]), markeredgecolor=POINT_TEST_EDGE if test else "k", markeredgewidth=cfg["point_edge"])
    if test:
        return dict(markerfacecolor="white", markeredgewidth=2.5)
    return dict(markerfacecolor=col, markeredgecolor="k", markeredgewidth=edge)


def save_legend(handles, out_dir, base, formats, dpi, fontsize, ncol=1):
    """凡例だけを別の画像（out_dir/<base>.<形式>）に保存する。保存したファイルの一覧を返す"""
    import matplotlib.pyplot as plt
    os.makedirs(out_dir, exist_ok=True)
    lf, paths = plt.figure(figsize=(6, 6)), []
    lf.legend(handles=handles, loc="center", fontsize=fontsize, framealpha=1.0, edgecolor="black", ncol=ncol)
    for fmt in formats:
        path = os.path.join(out_dir, f"{base}.{fmt}")
        lf.savefig(path, bbox_inches="tight", dpi=dpi)
        paths.append(path)
    plt.close(lf)
    return paths


PARITY_DEFAULT = dict(figsize=(9.5, 9), dpi=None, font={"axis": 10, "legend": 10, "title": 12, "tick": 10},
                      groups={"低速側": ("#1f77b4", "o"), "高速側": ("#d62728", "s"), "範囲内": ("#2ca02c", "^")},
                      marker=6, alpha=0.8, point_color=True, point_edge=1.5, point_edge_color="black", point_fill="white", point_fill_alpha=1.0,
                      errbar=True, err_color="0.3", err_line=1.2, err_alpha=1.0, err_on_top=False, capsize=3, diag_line=1.0,
                      lim=None, margin=1.3, aspect="equal", legend_separate=False, legend_loc="upper left", show_title=True, show_xlabel=True, show_ylabel=True,
                      xlabel=r"実測の粘度 $\eta$ (Pa·s)", ylabel=r"予測の粘度 $\eta$ (Pa·s)", grid=True, formats=["png"],
                      same_style=None, same_label="3data")
# 3data のパリティ図（学習用のノートのセル14）の見た目の既定（2026-11-08 追加）。
# 2026-11-09 のユーザーの依頼で：95% 予測区間の縦線は、点の種類によらず同じ色 err_color（既定は濃い灰色 "0.3"）・同じ太さ err_line にした。
# 縦線は既定で点の後ろに描く（err_on_top = False。ユーザーの指定）。高速側は区間が狭く〔上限/下限 約 1.03〜1.5 倍〕、縦線が点に隠れることがある。True なら点の上に描く。
# same_style＝(色, 形) を渡すと、低速側・高速側を区別せず、すべての点を同じ形で描く。
# 点は、色付き・色なしとも、枠線（色 point_edge_color・太さ point_edge）を付ける（2026-11-09 のユーザーの依頼。それまで色付きの点には枠線がなかった）。
# aspect：グラフエリアの縦横比。"equal"＝両軸の 1 桁を同じ長さにする（範囲が同じなので正方形。今まで）／数＝縦 ÷ 横 の比（例 0.75）／None＝図の大きさいっぱい


def parity_plot(p, title, out_dir, base, show=None, **kw):
    """3data の予測のパリティ図（横軸：実測の粘度、縦軸：予測の粘度、縦線：95% 予測区間）を 1 枚保存する。
    p は予測値の表（列 eta・予測η・下限95・上限95・group）。見た目は kw（PARITY_DEFAULT のキー）で変えられる：
    点には、色付き・色なしとも、枠線（色 point_edge_color・太さ point_edge）を付ける。
    point_color が False なら、点を塗りつぶし（色 point_fill・濃さ point_fill_alpha。既定は白）と枠線にする（低速側・高速側は点の形で見分ける）。
    aspect は、グラフエリアの縦横比（"equal"・縦 ÷ 横 の数・None）。
    区間の縦線は、点の種類によらず、色 err_color・太さ err_line・濃さ err_alpha で、err_on_top が True なら点の上に描く。
    same_style が (色, 形) なら、低速側・高速側を区別せず、すべての点をその色と形で描く（凡例の名前は same_label）。
    legend_separate が True なら、凡例をグラフに付けず、凡例だけを <base>_凡例 に保存する。保存したファイルの一覧を返す"""
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    unknown = set(kw) - set(PARITY_DEFAULT)
    if unknown:
        raise ValueError(f"パリティ図の設定に、知らない名前があります：{sorted(unknown)}")
    cfg = {**PARITY_DEFAULT, **kw}
    ft = {**PARITY_DEFAULT["font"], **(cfg["font"] or {})}
    dpi = cfg["dpi"] or FIG_DPI
    fig, ax = plt.subplots(figsize=cfg["figsize"])
    same = cfg["same_style"] is not None
    groups = {cfg["same_label"]: cfg["same_style"]} if same else cfg["groups"]
    for g, (col, mk) in groups.items():
        m = np.ones(len(p), dtype=bool) if same else (p["group"] == g).to_numpy()
        if not m.any():
            continue
        x, y = p.loc[m, "eta"], p.loc[m, "予測η"]
        if cfg["point_color"]:                  # 色付き：その組の色（濃さ alpha）で塗る
            face = fill_rgba(col, cfg["alpha"])
        else:                                   # 色なし：塗りつぶし（既定は白）
            face = fill_rgba(cfg["point_fill"], cfg["point_fill_alpha"])
        ax.plot(x, y, ls="none", marker=mk, ms=cfg["marker"], markerfacecolor=face, markeredgecolor=cfg["point_edge_color"], markeredgewidth=cfg["point_edge"],
                zorder=3, label=f"{g}（{int(m.sum())} ショット）")     # どの点にも枠線（色 point_edge_color・太さ point_edge）を付ける
        if cfg["errbar"]:                       # 95% 予測区間の縦線（点の種類によらず同じ色・太さ）
            ax.errorbar(x, y, yerr=[y - p.loc[m, "下限95"], p.loc[m, "上限95"] - y], fmt="none", ecolor=cfg["err_color"], elinewidth=cfg["err_line"],
                        capsize=cfg["capsize"], capthick=cfg["err_line"], alpha=cfg["err_alpha"], zorder=4 if cfg["err_on_top"] else 2)
    if cfg["lim"] is not None:
        lim = list(cfg["lim"])
    else:
        lim = [min(p["eta"].min(), p["下限95"].min()) / cfg["margin"], max(p["eta"].max(), p["上限95"].max()) * cfg["margin"]]
    ax.plot(lim, lim, "k--", lw=cfg["diag_line"], label="予測 = 実測")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    if cfg["aspect"] == "equal":                # 両軸の 1 桁を同じ長さに（範囲が同じなので、グラフエリアは正方形）
        ax.set_aspect("equal")
    elif cfg["aspect"] is not None:             # グラフエリアの 縦 ÷ 横 の比を指定
        ax.set_box_aspect(float(cfg["aspect"]))
    log_ticks(ax)
    ax.tick_params(axis="both", which="major", labelsize=ft["tick"])
    if cfg["show_xlabel"]:
        ax.set_xlabel(cfg["xlabel"], fontsize=ft["axis"])
    if cfg["show_ylabel"]:
        ax.set_ylabel(cfg["ylabel"], fontsize=ft["axis"])
    if cfg["show_title"]:
        ax.set_title(title, fontsize=ft["title"])
    if cfg["grid"]:
        ax.grid(True, which="both", ls="--", alpha=0.4)
    handles, _ = ax.get_legend_handles_labels()
    if cfg["errbar"]:                           # 凡例に、区間の縦線の記号を足す
        handles.append(Line2D([], [], ls="none", marker="|", ms=cfg["marker"] * 1.6, mew=cfg["err_line"], color=cfg["err_color"], alpha=cfg["err_alpha"], label="95% 予測区間"))
    if not cfg["legend_separate"]:
        ax.legend(handles=handles, loc=cfg["legend_loc"], fontsize=ft["legend"])
    os.makedirs(out_dir, exist_ok=True)
    saved = []
    for fmt in cfg["formats"]:
        path = os.path.join(out_dir, f"{base}.{fmt}")
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
        saved.append(path)
    if SHOW_FIGS if show is None else show:
        plt.show()
    plt.close(fig)
    if cfg["legend_separate"]:
        saved += save_legend(handles, out_dir, base + "_凡例", cfg["formats"], dpi, ft["legend"])
    return saved


T_CURVE_COLORS = ["#d62728", "#1f77b4", "#2ca02c", "#9467bd", "#8c564b"]       # 総合図の、加熱シリンダ温度ごとの線・帯の色（低い温度から：赤・青・緑）。点の色は厚みごと（赤・青・黄・緑）なので、色は同じでも、点は形で見分ける


def flow_plot_common(fit, mat, pool, train_conds, out_dir, stem, tr=None, **kw):
    """6月のノートのセル9-2 と同じ仕様（図の大きさ〔既定 12 × 10 インチ。6月は 15 × 10〕、文字の大きさ、点の大きさ〔既定 300〕、軸の目盛り〔両軸とも 10 の累乗〕、凡例の並べ方）で、
    全キャビティ厚みの**学習データ**の点（色・形は厚み）を重ね、**厚みによらない共通の平均関数と 95% 区間**を描く。検証（3data）の点は描かない。
    曲線は、せん断速度だけを動かし、ほかの入力を固定したもの：P3_3 は学習データ（tr）の中央値、Tff は、その設定温度の学習データの中央値、Tset はその温度。
    曲線の範囲は、その設定温度の学習データの範囲を extend 倍に延ばしたもの（x_range があればそれ）。軸の範囲は、common_xlim・common_ylim（None なら、データを含む 10 の累乗の範囲）。
    2 種類の図を保存する：①加熱シリンダ温度ごとに 1 枚（…_全厚み共通_<温度>C）、②全温度を 1 枚に重ね、温度ごとに線・帯の色を変えた総合図（…_全厚み共通_全温度）。
    保存したファイルの一覧を返す"""
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    cfg = {**FLOW_DEFAULT, "common_xlim": None, "common_ylim": None, "legend_ncol": 1, **kw}
    dpi = cfg["dpi"] or FIG_DPI
    tr = pool if tr is None else tr
    ft, saved = cfg["font"], []
    band_name = "95% 予測区間" if cfg["band"] == "predict" else "95% 信頼区間"
    p_med = float(np.exp(tr["ln_P33"].median()) / 1e6)
    levels = sorted(tr["Tset"].unique())
    tcol = {float(t): T_CURVE_COLORS[i % len(T_CURVE_COLORS)] for i, t in enumerate(levels)}
    curves = {}
    for t, sub in tr.groupby("Tset"):
        lgc = sub.groupby("cond")["ln_gd"].mean()
        lo, hi = (np.log(cfg["x_range"][0]), np.log(cfg["x_range"][1])) if cfg["x_range"] else (lgc.min() - np.log(cfg["extend"]), lgc.max() + np.log(cfg["extend"]))
        lg = np.linspace(lo, hi, cfg["n_grid"])
        tff_med = float(np.exp(sub["ln_Tff"].median()) - 273.15)
        mu, sd, sdf, _ = predict(fit, model_inputs(np.exp(lg), p_med, tff_med, float(t)))
        curves[float(t)] = (np.exp(lg), mu, sd if cfg["band"] == "predict" else sdf, tff_med)

    def points_of(sub):
        """学習データの点（px, py, 面積）。cfg["points"] が "cond" なら条件平均、"shot" ならショットごと"""
        if cfg["points"] == "cond":
            a = sub.groupby("cond").agg(lg=("ln_gd", "mean"), le=("ln_eta", "mean")).reset_index()
            return np.exp(a["lg"].to_numpy()), np.exp(a["le"].to_numpy()), cfg["marker"]
        return sub["gd"].to_numpy(), sub["eta"].to_numpy(), cfg["marker_shot"]

    def thickness_points(ax, sub, xs, ys, z=14):
        """厚みごとに色・形を変えて、点を描く（6月のセル9-2 と同じ：alpha 0.8、黒い縁）。凡例用の記号の一覧を返す"""
        hs = []
        for h in sorted(sub["H"].unique()):
            px, py, ms = points_of(sub[np.isclose(sub["H"], h)])
            col, mk = H_COLORS.get(float(h), "gray"), H_MARKS.get(float(h), "x")
            xs += list(px)
            ys += list(py)
            if cfg["point_color"]:
                ax.scatter(px, py, s=ms, color=col, marker=mk, alpha=0.8, edgecolors="k", linewidths=0.5, zorder=z)
            else:                                                 # 色なし：白の塗りつぶしと黒い枠線（太さ point_edge）
                ax.scatter(px, py, s=ms, marker=mk, zorder=z, **point_kw(cfg, col))
            hs.append(Line2D([], [], color=col, ls="none", marker=mk, markersize=np.sqrt(cfg["marker"]), label=f"{h:g} mm", **point_handle_kw(cfg, col, edge=0.5)))
        return hs

    def decade_lim(v, user):
        if user is not None:
            return user
        return 10.0 ** np.floor(np.log10(min(v))), 10.0 ** np.ceil(np.log10(max(v)))

    def save(fg, base):
        os.makedirs(out_dir, exist_ok=True)
        for fmt in cfg["formats"]:
            path = os.path.join(out_dir, f"{base}.{fmt}")
            fg.savefig(path, bbox_inches="tight", dpi=dpi)
            saved.append(path)

    def draw_curve(ax, t, tff_med, col, lw, xl):
        """平均関数と 95% 区間を、グラフエリアの端（横軸の範囲 xl）まで描く（6月のセル9-2 と同じ。軸の範囲を変えても、常に端まで描く）"""
        lg = np.linspace(np.log(xl[0]), np.log(xl[1]), max(cfg["n_grid"], 400))
        mu, sd, sdf, _ = predict(fit, model_inputs(np.exp(lg), p_med, tff_med, float(t)))
        s, x = (sd if cfg["band"] == "predict" else sdf), np.exp(lg)
        ax.fill_between(x, np.exp(mu - 1.96 * s), np.exp(mu + 1.96 * s), color=col, alpha=0.25, zorder=15)
        ax.plot(x, np.exp(mu - 1.96 * s), color=col, lw=0.5, alpha=0.6, zorder=16)
        ax.plot(x, np.exp(mu + 1.96 * s), color=col, lw=0.5, alpha=0.6, zorder=16)
        ax.plot(x, np.exp(mu), color=col, lw=lw, zorder=20)

    def finish(fig, ax, xs, ys, base, title, handles, separate_legend=False, handles_en=None, draw=None, keep_title=False):
        """軸・目盛り・凡例・タイトルを整えて保存する。separate_legend が True なら、グラフには凡例とタイトルを付けず（keep_title が True ならタイトルは付ける）、凡例だけを別の画像（base + "_凡例"）に保存する。
        draw：軸の範囲を決めた後で、曲線と帯をその範囲いっぱいに描く関数 draw(ax, 横軸の範囲)"""
        ax.tick_params(axis="both", which="both", direction="in", top=True, right=True, labelsize=ft["tick"], pad=8)
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlim(*decade_lim(xs, cfg["common_xlim"]))
        ax.set_ylim(*decade_lim(ys, cfg["common_ylim"]))
        if draw is not None:
            xl, yl = ax.get_xlim(), ax.get_ylim()
            draw(ax, xl)
            ax.set_xlim(*xl)
            ax.set_ylim(*yl)
        for axis in (ax.xaxis, ax.yaxis):                        # 6月のセル9-2 と同じ：両軸とも 10 の累乗の目盛り
            axis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
            axis.set_major_formatter(ticker.LogFormatterSciNotation())
            axis.set_minor_formatter(ticker.NullFormatter())
        ax.grid(True, which="major", linestyle="-", alpha=0.6, color="gray", zorder=1)
        ax.grid(True, which="minor", linestyle=":", alpha=0.4, zorder=1)
        if not separate_legend:
            ax.legend(handles=handles, loc="lower left", fontsize=ft["legend"], framealpha=1.0, edgecolor="black", ncol=cfg["legend_ncol"])
        if (not separate_legend or keep_title) and cfg["show_title"] and title:
            ax.set_title(title, fontsize=ft["title"], fontweight="bold", pad=15)
        if cfg["show_xlabel"]:
            ax.set_xlabel(r"せん断速度 $\dot{\gamma}$ (1/s)", fontsize=ft["axis"], fontweight="bold", labelpad=15)
        if cfg["show_ylabel"]:
            ax.set_ylabel(r"樹脂粘度 $\eta$ (Pa·s)", fontsize=ft["axis"], fontweight="bold", labelpad=15)
        fig.tight_layout()
        save(fig, base)
        if SHOW_FIGS if cfg["show"] is None else cfg["show"]:
            plt.show()
        plt.close(fig)
        if separate_legend:                                       # 凡例だけの画像
            lf = plt.figure(figsize=(6, 6))
            lf.legend(handles=handles, loc="center", fontsize=ft["legend"], framealpha=1.0, edgecolor="black", ncol=cfg["legend_ncol"])
            save(lf, base + "_凡例")
            plt.close(lf)
            if handles_en is not None:                            # 凡例の日本語を英語にした版（…_凡例_English）
                lf = plt.figure(figsize=(6, 6))
                lf.legend(handles=handles_en, loc="center", fontsize=ft["legend"], framealpha=1.0, edgecolor="black", ncol=cfg["legend_ncol"])
                save(lf, base + "_凡例_English")
                plt.close(lf)

    # ① 加熱シリンダ温度ごと（曲線は赤。6月のセル9-2 と同じ）
    for t, (x, mu, s, tff_med) in curves.items():
        fig, ax = plt.subplots(figsize=cfg["figsize"])
        xs, ys = list(x), list(np.exp(mu))                       # 軸の範囲を決めるための値（データの範囲の曲線と点）。曲線そのものは、finish が軸の端まで描く
        hs = thickness_points(ax, tr[tr["Tset"] == t], xs, ys)
        head = [Line2D([], [], color="red", lw=cfg["line"], label="予測平均値"), Patch(facecolor="red", alpha=0.25, label=band_name), Patch(color="none", label="キャビティ厚み")]
        note = []
        if cfg["legend_separate"]:                               # 凡例を別の画像にするときは、固定した値も凡例の画像に書く（グラフ本体には何も重ねない）
            note = [Patch(color="none", label="固定した値（学習データの中央値）"), Patch(color="none", label=f"　P3_3：{p_med:.2f} MPa"),
                    Patch(color="none", label=f"　Tff：{tff_med:.1f} ℃")]
        else:
            ax.text(0.98, 0.97, f"固定した値\nP3_3：{p_med:.2f} MPa\nTff：{tff_med:.1f} ℃", transform=ax.transAxes, ha="right", va="top", fontsize=ft["legend"] * 0.75,
                    bbox=dict(boxstyle="round", facecolor="white", edgecolor="black"), zorder=30)
        finish(fig, ax, xs, ys, f"{stem}_全厚み共通_{int(t)}C",
               f"{MAT_NAMES[mat]}　せん断速度と粘度の関係（設定温度 {int(t)} ℃）\n{model_title(fit['key'], fit.get('mode'))}", head + hs + note,
               separate_legend=cfg["legend_separate"], keep_title=True,
               draw=lambda a, xl, t=t, tm=tff_med: draw_curve(a, t, tm, "red", cfg["line"], xl))

    # ② 全温度を 1 枚に重ねた総合図（線・帯の色は温度、点の色・形は厚み）
    fig, ax = plt.subplots(figsize=cfg["figsize"])
    xs, ys = [], []
    for t, (x, mu, s, tff_med) in curves.items():
        xs += list(x)
        ys += list(np.exp(mu))
    hs = thickness_points(ax, tr, xs, ys)
    head = [Line2D([], [], color=tcol[t], lw=cfg["line"], label=f"予測平均値_{int(t)} ℃") for t in curves]
    head += [Patch(facecolor=tcol[t], alpha=0.25, label=f"{band_name}_{int(t)} ℃") for t in curves]
    head += [Patch(color="none", label="キャビティ厚み")]
    note = [Patch(color="none", label="固定した値（学習データの中央値）"), Patch(color="none", label=f"　P3_3：{p_med:.2f} MPa")]
    note += [Patch(color="none", label=f"　Tff（{int(t)} ℃）：{v[3]:.1f} ℃") for t, v in curves.items()]
    band_en = "95% prediction interval" if cfg["band"] == "predict" else "95% confidence interval"           # 凡例の英語版（hs の「0.5 mm」などは、そのまま）
    head_en = [Line2D([], [], color=tcol[t], lw=cfg["line"], label=f"Predicted mean_{int(t)} °C") for t in curves]
    head_en += [Patch(facecolor=tcol[t], alpha=0.25, label=f"{band_en}_{int(t)} °C") for t in curves]
    head_en += [Patch(color="none", label="Cavity thickness")]
    note_en = [Patch(color="none", label="Fixed values (median of training data)"), Patch(color="none", label=f"  P3_3: {p_med:.2f} MPa")]
    note_en += [Patch(color="none", label=f"  Tff ({int(t)} °C): {v[3]:.1f} °C") for t, v in curves.items()]
    finish(fig, ax, xs, ys, f"{stem}_全厚み共通_全温度", None, head + hs + note, separate_legend=True, handles_en=head_en + hs + note_en,
           draw=lambda a, xl: [draw_curve(a, t, v[3], tcol[t], cfg["line"], xl) for t, v in curves.items()])
    return saved


def flow_plot(fit, mat, pool, train_conds, out_dir, stem, **kw):
    """せん断速度 ― 粘度の学習結果グラフ（6月のノートのセル9-2 と同じ見せ方。本体の nb_common.hr_flow を、このデータ用に直したもの）。
    "temperature"：設定温度ごとに 1 枚（厚みごとの線）／"thickness"：厚みごとに 1 枚（設定温度ごとの線）。
    塗りつぶしの点は学習（5data）、白抜きは検証（3data）、× は問題データ一覧の「感度確認」。保存したファイルの一覧を返す"""
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    cfg = {**FLOW_DEFAULT, **kw}
    dpi = cfg["dpi"] or FIG_DPI
    ts = t_style(pool["Tset"].unique())
    curves = {}
    for (h, t), sub in pool.groupby(["H", "Tset"]):
        lgc = sub.groupby("cond")["ln_gd"].mean()
        if len(lgc) < 2:
            continue
        lo, hi = (np.log(cfg["x_range"][0]), np.log(cfg["x_range"][1])) if cfg["x_range"] else (lgc.min() - np.log(cfg["extend"]), lgc.max() + np.log(cfg["extend"]))
        lg = np.linspace(lo, hi, cfg["n_grid"])
        mu, sd, sdf, _ = predict(fit, flow_inputs(sub, lg, h, t))
        curves[(float(h), float(t))] = (np.exp(lg), mu, sd if cfg["band"] == "predict" else sdf)
    saved, ft = [], cfg["font"]
    band_name = "95% 予測区間" if cfg["band"] == "predict" else "95% 信頼区間"
    for layout_ in cfg["layouts"]:
        by_h = layout_ == "thickness"
        for gv in sorted({k[0] if by_h else k[1] for k in curves}):
            fig, ax = plt.subplots(figsize=cfg["figsize"])
            series, any_test, any_sens, xs, ys, pend = [], False, False, [], [], []
            for sv in sorted({(k[1] if by_h else k[0]) for k in curves if (k[0] if by_h else k[1]) == gv}):
                h, t = (gv, sv) if by_h else (sv, gv)
                col, mk = ts[t] if by_h else (H_COLORS.get(h, "gray"), H_MARKS.get(h, "x"))
                x, mu, s = curves[(float(h), float(t))]             # データの範囲の曲線。軸の範囲を決めるのに使う（曲線そのものは、軸の範囲が決まってから、端まで描く）
                sub = pool[np.isclose(pool["H"], h) & (pool["Tset"] == t)]
                pend.append((h, t, col, sub))
                if cfg["points"] == "cond":
                    a = sub.groupby("cond").agg(lg=("ln_gd", "mean"), le=("ln_eta", "mean"), sens=("is_sens", "max")).reset_index()
                    px, py, sens, ms = np.exp(a["lg"].to_numpy()), np.exp(a["le"].to_numpy()), a["sens"].to_numpy().astype(bool), cfg["marker"]
                    trm = a["cond"].isin(train_conds).to_numpy()
                else:
                    px, py, sens, ms = sub["gd"].to_numpy(), sub["eta"].to_numpy(), sub["is_sens"].to_numpy().astype(bool), cfg["marker_shot"]
                    trm = sub["cond"].isin(train_conds).to_numpy()
                xs += list(x) + list(px)
                ys += list(np.exp(mu)) + list(py)
                ax.scatter(px[trm], py[trm], s=ms, marker=mk, zorder=14, **point_kw(cfg, col))
                if cfg["show_test"] and (~trm).any():
                    ax.scatter(px[~trm], py[~trm], s=ms, marker=mk, zorder=14, **point_kw(cfg, col, test=True))
                    any_test = True
                if cfg["mark_sens"] and sens.any():
                    ax.scatter(px[sens], py[sens], s=ms * 1.3, color="k", marker="x", linewidths=3, zorder=25)
                    any_sens = True
                series.append((sv, col, mk))
            head = "キャビティ厚み" if not by_h else "設定温度"
            handles = [Patch(facecolor="gray", alpha=0.3, label=band_name), Patch(color="none", label=head)]
            handles += [Line2D([], [], color=c, lw=cfg["line"], marker=m, markersize=np.sqrt(cfg["marker"]) * 0.8, **point_handle_kw(cfg, c),
                               label=(f"{sv:g} mm" if not by_h else f"{int(sv)} ℃")) for sv, c, m in series]
            if any_test:
                test_lab = "検証（3data。白抜き）" if cfg["point_color"] else "検証（3data。灰色の枠）"
                handles.append(Line2D([], [], color="gray", ls="none", marker="o", markersize=np.sqrt(cfg["marker"]) * 0.8, label=test_lab, **point_handle_kw(cfg, "gray", test=True)))
            if any_sens:
                handles.append(Line2D([], [], color="k", ls="none", marker="x", markersize=np.sqrt(cfg["marker"]) * 0.8, markeredgewidth=3, label="感度確認の条件"))
            if cfg["band"] not in ("predict", "latent"):
                handles = handles[1:]
            if not cfg["legend_separate"]:
                ax.legend(handles=handles, loc="upper right", fontsize=ft["legend"], framealpha=1.0, edgecolor="black")
            ax.tick_params(axis="both", which="both", direction="in", top=True, right=True, labelsize=ft["tick"], pad=8)
            ax.set_xscale("log")
            ax.set_yscale("log")
            xl = cfg["xlim"][layout_] if isinstance(cfg["xlim"], dict) else cfg["xlim"]
            yl = cfg["ylim"][layout_] if isinstance(cfg["ylim"], dict) else cfg["ylim"]
            ax.set_xlim(*((min(xs) / 1.4, max(xs) * 1.4) if xl is None else xl))
            ax.set_ylim(*((min(ys) / 1.5, max(ys) * 1.5) if yl is None else yl))
            xlim_now, ylim_now = ax.get_xlim(), ax.get_ylim()
            lg_full = np.linspace(np.log(xlim_now[0]), np.log(xlim_now[1]), max(cfg["n_grid"], 400))       # 平均関数と 95% 区間を、グラフエリアの端まで描く（軸の範囲を変えても追従）
            for h, t, col, sub in pend:
                mu_f, sd_f, sdf_f, _ = predict(fit, flow_inputs(sub, lg_full, h, t))
                s_f, x_f = (sd_f if cfg["band"] == "predict" else sdf_f), np.exp(lg_full)
                if cfg["band"] in ("predict", "latent"):
                    ax.fill_between(x_f, np.exp(mu_f - 1.96 * s_f), np.exp(mu_f + 1.96 * s_f), color=col, alpha=0.22, zorder=15)
                ax.plot(x_f, np.exp(mu_f), color=col, lw=cfg["line"], zorder=20)
            ax.set_xlim(*xlim_now)
            ax.set_ylim(*ylim_now)
            for axis, auto in ((ax.xaxis, xl is None), (ax.yaxis, yl is None)):
                lo_, hi_ = axis.get_view_interval()
                fine = auto and np.log10(hi_ / lo_) <= 2.5                 # 範囲を自動で決めた軸で、2.5 桁以下なら 1・2・5 × 10ⁿ の目盛り（log_ticks と同じ基準）
                if fine:
                    axis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0, 2.0, 5.0)))
                    axis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:g}"))
                    fine = not tick_labels_overlap(fig, axis)            # それでも目盛りの数字が重なるなら、10ⁿ だけにする
                if not fine:
                    axis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
                    axis.set_major_formatter(ticker.LogFormatterSciNotation())
                axis.set_minor_formatter(ticker.NullFormatter())
            ax.grid(True, which="major", linestyle="-", alpha=0.6, color="gray", zorder=1)
            ax.grid(True, which="minor", linestyle=":", alpha=0.4, zorder=1)
            gtxt = f"厚み {gv:g} mm" if by_h else f"設定温度 {int(gv)} ℃"
            if cfg["show_title"]:
                ax.set_title(f"{MAT_NAMES[mat]}　せん断速度と粘度の関係（{gtxt}）\n{model_title(fit['key'], fit.get('mode'))}", fontsize=ft["title"], pad=15)
            if cfg["show_xlabel"]:
                ax.set_xlabel(r"せん断速度 $\dot{\gamma}$ (1/s)", fontsize=ft["axis"], labelpad=15)
            if cfg["show_ylabel"]:
                ax.set_ylabel(r"樹脂粘度 $\eta$ (Pa·s)", fontsize=ft["axis"], labelpad=15)
            fig.tight_layout()
            base = f"{stem}_{'厚みごと' if by_h else '設定温度ごと'}_{(f'{gv:g}mm') if by_h else (str(int(gv)) + 'C')}"
            os.makedirs(out_dir, exist_ok=True)
            for fmt in cfg["formats"]:
                path = os.path.join(out_dir, f"{base}.{fmt}")
                fig.savefig(path, bbox_inches="tight", dpi=dpi)
                saved.append(path)
            if SHOW_FIGS if cfg["show"] is None else cfg["show"]:
                plt.show()
            plt.close(fig)
            if cfg["legend_separate"]:                             # 凡例だけの画像（<図の名前>_凡例）
                saved += save_legend(handles, out_dir, base + "_凡例", cfg["formats"], dpi, ft["legend"])
    return saved

### Tff を入力から外した C（C_noTff）を足す

共通の部品は変えず、このノートの中だけで、モデルの記号 `C_noTff` を足す（GP の入力の列を決める `layout` を包み、`C_noTff` のときだけ、C から `ln_Tff` を外した列を返す）。学習・予測・ハイパーパラメータの表・図は、共通の部品の関数がそのまま使える。

In [ ]:
# セル2a：C_noTff（C から流動中樹脂温度 Tff を外したモデル）を足す（このノートの中だけ。mat_common.py は変えない）
NO_TFF = "C_noTff"
_layout_common = layout


def layout(key):
    """GP の入力の列と、シフトに使う温度の列。C_noTff だけ、C の入力から ln Tff を外したもの（ln γ̇, ln P3_3, Tset。シフトなし）。ほかのモデルは共通の部品のまま"""
    if key == NO_TFF:
        return ["ln_gd", "ln_P33", "Tset"], None
    return _layout_common(key)


MODEL_JA[NO_TFF] = "C_noTff（シフト構造なし。C から流動中樹脂温度 Tff を外し、3 変数を並べた通常の GPR）"
MODEL_DIRS[NO_TFF] = "C_noTff_温度シフトなし_型内樹脂温度を入力に使わない"
print("C の入力：" + "、".join(input_labels("C")))
print("C_noTff の入力：" + "、".join(input_labels(NO_TFF)))

## 2. 検討の結果とデータの読み込み

**このセルで、材料ごとにダイアログが 1 回開く**（シフトファクター検討用のノートで選んだ学習結果フォルダを選ぶ。設定 `FILES` に書いたものは開かない）。検討の結果（学習済みの C、3data の予測値、表）を読み、学習済みの C に記録された学習データ・検証データを、シフトファクター検討用のノートと同じ手順で読む（問題データ一覧の「除外」を除く）。データの版（通常版／温度適切版）は、学習データのフォルダの名前（「温度適切」を含むか）で表示する。

**判定**：学習済みの C の入力の取り方が `INPUT_MODE` と同じ／読んだ C で 3data を予測し直すと、保存済みの予測値と一致する（データとモデルの組み合わせが正しい）／C の LML が `ハイパーパラメータ.csv` の値と一致する。

In [ ]:
# セル3：読み込み
import joblib
JUDGES.clear()
display(choose_inputs(MATERIALS, FILES, need_data=False))      # 材料ごとに、学習結果フォルダを決める（設定 FILES に書いていなければ、ダイアログで選ぶ）
OUT_NAME = "Tffの入力の有無の比較"
STUDY_CSVS = {"hyper": "ハイパーパラメータ.csv", "est": "シフトファクターの推定.csv", "metrics": "予測検証_3data_指標.csv"}   # 検討のノートの表（読むだけ）


def study_dir(mat):
    """シフトファクター検討用のノートの結果のフォルダ：<学習結果フォルダ>/<版>/全厚み総合/シフトファクターの検討/（読むだけなので、フォルダは作らない）"""
    return result_dir(mat, VERSION_DIRS[INPUT_MODE], THICK_TOTAL, STUDY_DIR, make=False)


def out_dir(mat, sub=None):
    """このノートの結果の置き場所：<学習結果フォルダ>/<版>/全厚み総合/Tffの入力の有無の比較/（SAVE_ROOT を書いたときは、その下の <材料>/<版>/全厚み総合/…）"""
    base = os.path.join(full_path(SAVE_ROOT), mat, VERSION_DIRS[INPUT_MODE], THICK_TOTAL) if SAVE_ROOT else result_dir(mat, VERSION_DIRS[INPUT_MODE], THICK_TOTAL, make=False)
    d = os.path.join(base, OUT_NAME, *([sub] if sub else []))
    os.makedirs(d, exist_ok=True)
    return d


OBJ_C, FIT_C, PRED_C, TAB, DATA_VER = {}, {}, {}, {}, {}
for mat in MATERIALS:
    for name, fn in STUDY_CSVS.items():
        p = os.path.join(study_dir(mat), fn)
        if not os.path.exists(p):
            raise FileNotFoundError(f"{p} がありません。先にシフトファクター検討用のノートを、INPUT_MODE = '{INPUT_MODE}' で実行し、同じ学習結果フォルダを選んでください")
        TAB[(mat, name)] = pd.read_csv(p, encoding="utf-8-sig")
    path = os.path.join(study_dir(mat), MODEL_DIRS["C"], "学習済みモデル_C.joblib")
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} がありません（シフトファクター検討用のノートで、基準の C を学習していない）")
    OBJ_C[mat] = joblib.load(path)
    if OBJ_C[mat]["材料"] != MAT_NAMES[mat]:
        raise ValueError(f"{path} は {OBJ_C[mat]['材料']} のモデルです（{MAT_NAMES[mat]} の学習結果フォルダを選んでください）")
    FIT_C[mat] = OBJ_C[mat]["fit"]
    PRED_C[mat] = pd.read_csv(os.path.join(study_dir(mat), MODEL_DIRS["C"], "予測検証_3data_予測値.csv"), encoding="utf-8-sig")
    data_from_model(mat, OBJ_C[mat])         # 学習データ・検証データの場所は、学習済みの C の記録から取る（設定 FILES に書いてあれば、そちら）

D = load_all(MATERIALS)
TRAIN, TEST, POOL = {}, {}, {}
for mat in MATERIALS:
    TRAIN[mat] = D[(mat, "5data")]
    TEST[mat] = add_groups(TRAIN[mat], D[(mat, "3data")])
    POOL[mat] = pd.concat([TRAIN[mat], TEST[mat]], ignore_index=True)
    DATA_VER[mat] = "温度適切版" if "温度適切" in os.path.basename(os.path.dirname(SOURCES[mat]["5data"])) else "通常版"
    print(f"【{MAT_NAMES[mat]}・{DATA_VER[mat]}】学習データ：{show_path(SOURCES[mat]['5data'])}（{len(TRAIN[mat])} 行・{TRAIN[mat]['cond'].nunique()} 条件）"
          f"／検証データ：{show_path(SOURCES[mat]['3data'])}（{len(TEST[mat])} 行）")
    judge(FIT_C[mat].get("mode") == INPUT_MODE, f"学習済みの C の入力の取り方が {INPUT_MODE}", mat, "2 読み込み")
    mu, _, _, _ = predict(FIT_C[mat], TEST[mat])
    chk = TEST[mat][["file"]].assign(mu=mu).merge(PRED_C[mat][["file", "予測_ln_eta"]], on="file")
    judge(len(chk) == len(TEST[mat]) and np.allclose(chk["mu"], chk["予測_ln_eta"], atol=1e-6),
          f"C で 3data を予測し直した値が、保存済みの予測値と一致（{len(chk)}/{len(TEST[mat])} 行）", mat, "2 読み込み")
    h = TAB[(mat, "hyper")].set_index("model")
    lml_c = float(FIT_C[mat]["gp"].log_marginal_likelihood_value_)
    judge("C" in h.index and abs(lml_c - float(h.loc["C", "LML"])) < 1e-6, f"学習済みの C の LML（{lml_c:.2f}）が、ハイパーパラメータ.csv の値と一致", mat, "2 読み込み")
    print(f"  検討のモデル：{'、'.join(h.index)}")

## 3. C_noTff の学習

材料ごとに、C と同じ 5data・同じカーネル・同じ設定（`N_RESTARTS` 回のやり直し、乱数シード）で、入力から ln Tff を外したモデル C_noTff を学習する（シフトなし）。学習済みモデルは、結果のフォルダに保存する。

**判定**：C_noTff の LML が、検討のノートの θ = 0 の LML（A_Tff・B_Tff の `LML_at_0`。同じモデルを、プロファイル尤度の点の解から念入りに最適化した値）より `LML_CHECK` 以上低くない（低ければ、最適化が局所解に止まったおそれ。`N_RESTARTS` を増やして実行し直す）。

In [ ]:
# セル4：C_noTff の学習
from joblib import Parallel, delayed
import sklearn
t0 = time.time()
res = Parallel(n_jobs=min(N_JOBS, len(MATERIALS)))(
    delayed(fit_job)(NO_TFF, TRAIN[mat], FIT_C[mat]["cfg"], 0.0, 0.0, N_RESTARTS, None, INPUT_MODE) for mat in MATERIALS)
FIT_N = dict(zip(MATERIALS, res))
print(f"学習：{len(MATERIALS)} 材料、{(time.time() - t0) / 60:.1f} 分")
LML_REF = {}
for mat in MATERIALS:
    f = FIT_N[mat]
    lml = float(f["gp"].log_marginal_likelihood_value_)
    path = os.path.join(out_dir(mat), f"学習済みモデル_{NO_TFF}.joblib")
    joblib.dump({"fit": f, "材料": MAT_NAMES[mat], "モデル": NO_TFF, "説明": model_title(NO_TFF), "入力": input_labels(NO_TFF), "入力の取り方": INPUT_MODE,
                 "学習データのファイル": show_path(SOURCES[mat]["5data"]), "検証データのファイル": show_path(SOURCES[mat]["3data"]),
                 "scikit-learn": sklearn.__version__, "作成日": time.strftime("%Y-%m-%d")}, path)
    est = TAB[(mat, "est")]
    ref = est[est["モデル"].astype(str).str.contains("_Tff")]["LML_at_0"]
    LML_REF[mat] = float(ref.max()) if len(ref) else np.nan
    print(f"【{MAT_NAMES[mat]}・{DATA_VER[mat]}】C_noTff の LML {lml:.2f}（検討の θ = 0 の LML {LML_REF[mat]:.2f}）。保存：{show_path(path)}")
    if np.isfinite(LML_REF[mat]):
        judge(lml >= LML_REF[mat] - LML_CHECK, f"C_noTff の LML {lml:.2f} が、検討の θ = 0 の LML {LML_REF[mat]:.2f}（A_Tff・B_Tff の LML_at_0。同じモデル）より {LML_CHECK:g} 以上低くない", mat, "3 学習")
    else:
        print("  （検討のノートで Tff のシフトのモデルを学習していないので、θ = 0 の LML と照らし合わせられない）")

## 4. 対数周辺尤度（LML）の比較（同じ材料・同じ版の中だけ）

検討のすべてのモデル（`ハイパーパラメータ.csv` の LML。シフトのあるモデルは、最良の係数での念入りな最適化の値）と C_noTff を、1 つの表に並べる。

- **パラメータの数**：カーネルのハイパーパラメータ（Matern の振幅 1・長さスケール〔入力の数〕・線形項 2・ノイズ 1）＋ シフトの係数（A は 1、B は 2）。C は 8、C_noTff は 7、A は 8、B は 9。
- **C_noTff との差** = LML(モデル) − LML(C_noTff)。**基準** = パラメータの数の差 k に対する、カイ二乗分布の 95% 点 ÷ 2（k = 1 で 1.92、k = 2 で 3.0。モデルが入れ子でないときは目安）。差が基準以上なら、「そのモデルは、Tff を使わない C_noTff より、尤度で支持される」。
- **本番の候補**：検討のノートと同じ規則（`shift_candidates`。候補がなければ C）で、`シフトファクターの推定.csv` から決め直したもの。

**判定**（★ は「Tff が尤度に効いていない」ことの知らせ）：C（Tff あり）が C_noTff より基準以上大きい／本番の候補が C_noTff より基準以上大きい。

In [ ]:
# セル5：対数周辺尤度の比較
from scipy.stats import chi2


def n_params(key):
    """モデルのパラメータの数：カーネル（振幅 1 ＋ 長さスケール〔入力の数〕＋ 線形項 2〔使うとき〕＋ ノイズ 1）＋ シフトの係数（A 1・B 2・C 0）"""
    d = len(layout(key)[0])
    s = {"A": 1, "B": 2}.get(str(key).split("_")[0], 0)
    return 1 + d + (2 if USE_LINEAR else 0) + 1 + s


def lml_crit(k):
    """パラメータの数の差 k に対する、LML の差の基準（カイ二乗分布の 95% 点 ÷ 2。k = 1 で 1.92）"""
    return float(chi2.ppf(0.95, k) / 2) if k > 0 else 0.0


LML_TAB, CAND = {}, {}
for mat in MATERIALS:
    h = TAB[(mat, "hyper")]
    est = TAB[(mat, "est")]
    cand, _, notes = shift_candidates([{"モデル": r["モデル"], "LML": r["LML_best"], "dLML_vs_0": r["dLML_vs_0"]} for _, r in est.iterrows()])
    CAND[mat] = cand or "C"
    lml_n = float(FIT_N[mat]["gp"].log_marginal_likelihood_value_)
    rows = [{"モデル": r["model"], "LML": float(r["LML"])} for _, r in h.iterrows()] + [{"モデル": NO_TFF, "LML": lml_n}]
    t = pd.DataFrame(rows)
    t["GP の入力"] = ["、".join(input_labels(k)) for k in t["モデル"]]
    t["シフトに使う温度"] = [layout(k)[1] or "なし" for k in t["モデル"]]
    t["Tff を使うか"] = ["使わない" if k == NO_TFF else ("シフトに使う" if layout(k)[1] == "Tff" else "GP の入力に使う") for k in t["モデル"]]
    t["パラメータの数"] = [n_params(k) for k in t["モデル"]]
    t["C_noTff との差"] = t["LML"] - lml_n
    t["基準（目安）"] = [lml_crit(n - n_params(NO_TFF)) for n in t["パラメータの数"]]
    t["C_noTff より支持される"] = [None if k == NO_TFF else bool(d >= c) for k, d, c in zip(t["モデル"], t["C_noTff との差"], t["基準（目安）"])]
    t["本番の候補"] = t["モデル"] == CAND[mat]
    t.insert(0, "版", DATA_VER[mat])
    t.insert(0, "材料", MAT_NAMES[mat])
    LML_TAB[mat] = t
    to_csv(t, os.path.join(out_dir(mat), "対数周辺尤度の比較.csv"))
    print(f"\n【{MAT_NAMES[mat]}・{DATA_VER[mat]}】本番の候補（検討の規則）：{CAND[mat]}" + ("（注意：" + "／".join(notes) + "）" if notes else ""))
    display(t.drop(columns=["材料", "版"]).round(2))
    rc = t.set_index("モデル").loc["C"]
    judge(bool(rc["C_noTff より支持される"]), f"C（Tff を GP の入力に使う）が C_noTff（Tff を使わない）より尤度で支持される（差 {rc['C_noTff との差']:+.2f}、基準 {rc['基準（目安）']:.2f}）。"
          "★ なら、Tff を入力に足しても尤度は上がらない", mat, "4 尤度")
    if CAND[mat] != "C":
        rk = t.set_index("モデル").loc[CAND[mat]]
        judge(bool(rk["C_noTff より支持される"]), f"本番の候補 {CAND[mat]} が C_noTff より尤度で支持される（差 {rk['C_noTff との差']:+.2f}、基準 {rk['基準（目安）']:.2f}）。"
              "★ なら、Tff を使わない C_noTff と尤度で差がない", mat, "4 尤度")

## 5. 3data の予測の比較

C_noTff で 3data を予測し、検討のノートと同じ指標（`metrics_rows`：ln η の RMSE・R²・95% 予測区間に入る割合〔被覆率〕など。全体・低速側・高速側）を出して、検討のすべてのモデル（`予測検証_3data_指標.csv`）と並べる。RMSE は、材料や版をまたいでも同じ物差し（ln η。0.1 で粘度の約 10%）。

図：①モデルごとの RMSE の棒グラフ（全体・低速側・高速側。C は灰色、C_noTff は橙、本番の候補は太枠）、②C と C_noTff のパリティ図（実測と予測の粘度。縦の線は 95% 予測区間）。

**判定**：RMSE（全体）の比 C_noTff / C が 1/`RMSE_RATIO`〜`RMSE_RATIO` の中（★ なら、Tff を外すと誤差がはっきり変わる。比が大きければ Tff は誤差を減らしている、小さければ Tff を外したほうが誤差が小さい）／C_noTff の被覆率（全体）が 0.8 以上。

In [ ]:
# セル6：3data の予測の比較
PRED_N, MT_ALL, RATIO = {}, {}, {}
RANGES = ["全体", "低速側", "高速側"]
GCOL = {"低速側": ("#1f77b4", "o"), "高速側": ("#d62728", "s"), "範囲内": ("#2ca02c", "^")}
for mat in MATERIALS:
    te = TEST[mat]
    mu, sd, sdf, nsd = predict(FIT_N[mat], te)
    PRED_N[mat] = te[["file", "cond", "Tset", "H", "v", "group", "eta", "gd", "P33", "Tff", "ln_eta"]].assign(
        予測_ln_eta=mu, 予測標準偏差=sd, 潜在関数の標準偏差=sdf, 予測η=np.exp(mu), 下限95=np.exp(mu - 1.96 * sd), 上限95=np.exp(mu + 1.96 * sd))
    to_csv(PRED_N[mat], os.path.join(out_dir(mat), f"予測検証_3data_予測値_{NO_TFF}.csv"))
    mt = pd.concat([TAB[(mat, "metrics")], pd.DataFrame(metrics_rows(mat, NO_TFF, te, mu, sd, sdf, {"ノイズの標準偏差": nsd}))], ignore_index=True)
    mt.insert(1, "版", DATA_VER[mat])
    MT_ALL[mat] = mt
    to_csv(mt, os.path.join(out_dir(mat), "予測検証_3data_指標_全モデル.csv"))
    rng = [r for r in RANGES if r in set(mt["範囲"])]
    piv = mt.pivot_table(index="モデル", columns="範囲", values="RMSE", sort=False)[rng]
    cov = mt.pivot_table(index="モデル", columns="範囲", values="coverage95", sort=False)[rng]
    RATIO[mat] = (piv.loc[NO_TFF] / piv.loc["C"]).to_dict()
    print(f"\n【{MAT_NAMES[mat]}・{DATA_VER[mat]}】3data の ln η の RMSE（本番の候補：{CAND[mat]}）")
    display(piv.round(4))
    print("95% 予測区間に入る割合（被覆率）")
    display(cov.round(3))
    print("RMSE の比 C_noTff / C：" + "、".join(f"{r} {RATIO[mat][r]:.3f}" for r in rng))
    # ① RMSE の棒グラフ
    fig, axes = plt.subplots(1, len(rng), figsize=(6.5 * len(rng), 6), squeeze=False)
    for ax, r in zip(axes[0], rng):
        vals = piv[r]
        cols = ["#ff7f0e" if k == NO_TFF else ("#7f7f7f" if k == "C" else "#1f77b4") for k in vals.index]
        bars = ax.bar(range(len(vals)), vals.to_numpy(), color=cols)
        for b, k in zip(bars, vals.index):
            if k == CAND[mat]:
                b.set_edgecolor("k")
                b.set_linewidth(2.5)
        ax.set_xticks(range(len(vals)))
        ax.set_xticklabels(vals.index, rotation=60, ha="right")
        ax.set_ylabel("3data の RMSE（ln η）")
        ax.set_title(f"{r}")
        ax.grid(True, axis="y", ls="--", alpha=0.4)
    fig.suptitle(f"{MAT_NAMES[mat]}・{DATA_VER[mat]}　3data の RMSE（灰：C、橙：C_noTff、太枠：本番の候補 {CAND[mat]}）")
    fig.tight_layout(rect=(0, 0, 1, 0.94))              # 上に、全体のタイトルの場所を空ける
    save_fig(fig, out_dir(mat), "予測検証_RMSEの比較_全モデル")
    # ② パリティ図（C と C_noTff）
    fig, axes = plt.subplots(1, 2, figsize=(18, 9))
    for ax, (k, p) in zip(axes, [("C", PRED_C[mat]), (NO_TFF, PRED_N[mat])]):
        rr = mt[(mt["モデル"] == k) & (mt["範囲"] == "全体")].iloc[0]
        for g, (col, mk) in GCOL.items():
            m = (p["group"] == g).to_numpy()
            if m.any():
                ax.errorbar(p.loc[m, "eta"], p.loc[m, "予測η"], yerr=[p.loc[m, "予測η"] - p.loc[m, "下限95"], p.loc[m, "上限95"] - p.loc[m, "予測η"]],
                            fmt=mk, color=col, ms=6, elinewidth=1.2, capsize=3, alpha=0.8, label=f"{g}（{int(m.sum())} ショット）")
        lim = [min(p["eta"].min(), p["下限95"].min()) / 1.3, max(p["eta"].max(), p["上限95"].max()) * 1.3]
        ax.plot(lim, lim, "k--", lw=1, label="予測 = 実測")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlim(lim)
        ax.set_ylim(lim)
        ax.set_aspect("equal")
        log_ticks(ax)
        ax.set_xlabel(r"実測の粘度 $\eta$ (Pa·s)")
        ax.set_ylabel(r"予測の粘度 $\eta$ (Pa·s)")
        ax.set_title(f"{k}（Tff を{'使わない' if k == NO_TFF else '入力に使う'}）\nRMSE(ln η) {rr['RMSE']:.3f}、R²(実数) {rr['R2_real']:.4f}、被覆率 {rr['coverage95']:.2f}")
        ax.grid(True, which="both", ls="--", alpha=0.4)
        ax.legend(loc="upper left")
    fig.suptitle(f"{MAT_NAMES[mat]}・{DATA_VER[mat]}　3data のパリティ図：C と C_noTff", y=0.99)
    fig.tight_layout(rect=(0, 0, 1, 0.93))              # 上に、全体のタイトルの場所を空ける（各図のタイトルが 2 行なので広めに）
    save_fig(fig, out_dir(mat), "予測検証_3data_パリティ_CとC_noTff")
    q = RATIO[mat]["全体"]
    judge(1 / RMSE_RATIO <= q <= RMSE_RATIO, f"3data の RMSE（全体）の比 C_noTff / C = {q:.3f} が {1 / RMSE_RATIO:.3f}〜{RMSE_RATIO:g} の中（Tff を外しても、誤差は同等）。"
          "★ なら、比が大きいと Tff が誤差を減らしている、小さいと Tff を外したほうが誤差が小さい", mat, "5 検証")
    cv = float(cov.loc[NO_TFF, "全体"])
    judge(cv >= 0.8, f"C_noTff の 95% 予測区間に入る割合（全体）が 0.8 以上（{cv:.3f}。C は {float(cov.loc['C', '全体']):.3f}）", mat, "5 検証")

## 6. ハイパーパラメータ（C と C_noTff）

C（検討の `ハイパーパラメータ.csv`）と C_noTff の、振幅・長さスケール・線形項・ノイズ・LML を並べる。入力は学習データで標準化してあるので、長さスケールはデータの幅（約 ±2）と比べて読む（長いほど、その入力の方向に、Matern の部分がゆっくりしか変わらない。線形項は入力ごとの長さスケールを持たない〔全入力に共通〕ので、長さスケールが長くても、その入力が線形に効いている場合がある）。上限 100 に張り付いたものは、表の `ls_at_bound` が True。

In [ ]:
# セル7：ハイパーパラメータ
HP = {}
for mat in MATERIALS:
    rn = kernel_row(FIT_N[mat])
    th_n, tv_n, _ = theta_names(FIT_N[mat]["cfg"])
    rn[th_n], rn[tv_n] = np.nan, np.nan                     # C_noTff にシフトの係数はない
    hc = TAB[(mat, "hyper")]
    rc = hc[hc["model"] == "C"].iloc[0].to_dict()
    t = pd.DataFrame([rc, {"材料": MAT_NAMES[mat], **rn}]).dropna(axis=1, how="all")
    t.insert(1, "版", DATA_VER[mat])
    HP[mat] = t
    to_csv(t, os.path.join(out_dir(mat), "ハイパーパラメータ_CとC_noTff.csv"))
    print(f"\n【{MAT_NAMES[mat]}・{DATA_VER[mat]}】")
    display(t.set_index("model").drop(columns=["材料", "版"]).T)

## 7. C_noTff の学習結果グラフ（せん断速度 ― 粘度。厚みごと）

学習用・検討用のノートと同じ関数（`flow_plot`）で、厚みごとに 1 枚、設定温度ごとの予測線と 95% 予測区間を描く（塗りつぶしの点は学習〔5data〕、白抜きは検証〔3data〕。点はショットごとの計測値）。曲線は、P3_3 を条件ごとの値で動かした断面（Tff は入力にないので使わない）。検討のノートの C の図は、そのノートの結果のフォルダにある。

In [ ]:
# セル8：C_noTff の学習結果グラフ
n_saved = 0
for mat in MATERIALS:
    saved = flow_plot(FIT_N[mat], mat, POOL[mat], set(TRAIN[mat]["cond"]), out_dir(mat, "せん断速度_粘度"), f"せん断速度_粘度_{NO_TFF}",
                      layouts=["thickness"], points="shot", show=SHOW_FIGS)
    n_saved += len(saved)
print(f"保存した図：{n_saved} 枚（せん断速度_粘度/）")

## 8. まとめ：Tff を GP の入力に使ったほうがよいか

材料ごとに、次の 2 つから結論を出す（同じ材料・同じ版の中だけで比べる）。
- **尤度**：LML(C) − LML(C_noTff) が `LML_CLEAR`（1.92）以上なら「Tff ありが支持される」、そうでなければ「支持されない」。
- **誤差**：3data の RMSE（全体）の比 C_noTff / C が `RMSE_RATIO`（1.1）より大きければ「Tff ありが良い」、1/1.1 より小さければ「Tff なしが良い」、その間なら「同等」。

| 尤度 | 誤差 | 結論 |
|---|---|---|
| 支持される | 同等・Tff ありが良い | 入れたほうがよい |
| 支持されない | Tff ありが良い | 入れたほうがよい（誤差で） |
| 支持されない | 同等 | どちらでもよい（外しても精度は同じ） |
| 支持されない | Tff なしが良い | 外したほうがよい |
| 支持される | Tff なしが良い | 決まらない（★） |

参考に、本番の候補（Tset でシフトするモデルは、Tff を GP の入力に使う）と C_noTff の差も並べる。結果は `Tffの入力の有無_まとめ.csv`。

In [ ]:
# セル9：まとめ
SUM = []
for mat in MATERIALS:
    t = LML_TAB[mat].set_index("モデル")
    d_c = float(t.loc["C", "C_noTff との差"])
    q = RATIO[mat]
    lik = "支持される" if d_c >= LML_CLEAR else "支持されない"
    err = "Tff ありが良い" if q["全体"] > RMSE_RATIO else ("Tff なしが良い" if q["全体"] < 1 / RMSE_RATIO else "同等")
    concl = {("支持される", "同等"): "入れたほうがよい", ("支持される", "Tff ありが良い"): "入れたほうがよい",
             ("支持されない", "Tff ありが良い"): "入れたほうがよい（誤差で）", ("支持されない", "同等"): "どちらでもよい（外しても精度は同じ）",
             ("支持されない", "Tff なしが良い"): "外したほうがよい", ("支持される", "Tff なしが良い"): "決まらない（★）"}[(lik, err)]
    piv = MT_ALL[mat].pivot_table(index="モデル", columns="範囲", values="RMSE", sort=False)
    ck = CAND[mat]
    SUM.append({"材料": MAT_NAMES[mat], "版": DATA_VER[mat], "LML C": float(t.loc["C", "LML"]), "LML C_noTff": float(t.loc[NO_TFF, "LML"]),
                "LML の差 C − C_noTff": d_c, "尤度": lik,
                **{f"RMSE C {r}": float(piv.loc["C", r]) for r in RANGES if r in piv.columns},
                **{f"RMSE C_noTff {r}": float(piv.loc[NO_TFF, r]) for r in RANGES if r in piv.columns},
                "RMSE の比 C_noTff / C（全体）": q["全体"], "誤差": err, "結論": concl,
                "本番の候補": ck, "LML の差 本番の候補 − C_noTff": float(t.loc[ck, "C_noTff との差"]), "その基準（目安）": float(t.loc[ck, "基準（目安）"]),
                "RMSE の比 C_noTff / 本番の候補（全体）": float(piv.loc[NO_TFF, "全体"] / piv.loc[ck, "全体"])})
    judge(concl != "決まらない（★）", f"結論：{concl}（尤度：{lik}〔差 {d_c:+.2f}〕、誤差：{err}〔比 {q['全体']:.3f}〕）", mat, "8 まとめ")
ST = pd.DataFrame(SUM)
display(ST.round(4))
for mat in MATERIALS:
    to_csv(ST[ST["材料"] == MAT_NAMES[mat]], os.path.join(out_dir(mat), "Tffの入力の有無_まとめ.csv"))

## 9. 判定の一覧

★ の行を確かめる。材料ごとに、結果のフォルダの `判定の一覧.csv` に保存する。

In [ ]:
# セル10：判定の一覧
JT = pd.DataFrame(JUDGES)
print(f"判定 {len(JT)} 件のうち、★要確認 {int((JT['判定'] != 'OK').sum())} 件")
display(JT[JT["判定"] != "OK"])
for mat in MATERIALS:
    to_csv(JT[JT["材料"].isin([MAT_NAMES[mat], ""])], os.path.join(out_dir(mat), "判定の一覧.csv"))
print("結果の置き場所：" + "、".join(show_path(out_dir(m)) for m in MATERIALS))